# Disaster XAI-Net — ConvNeXt + ViT **Trained Feature-Fusion Ensemble**, 5-Fold CV + Held-Out Test

Two-stage satellite damage assessment. One model per stage, trained end-to-end on images:

```
            PRE tile/crop  +  POST tile/crop
                 |                    |
     ┌───────────┴──────────┐  ┌──────┴───────────────┐
     │ ConvNeXt-Tiny Siamese │  │ ViT-B/16 Siamese      │   (each: one shared-weight
     │ f_pre ⊕ f_post (1536) │  │ f_pre ⊕ f_post (1536) │    backbone over pre & post)
     └───────────┬──────────┘  └──────┬───────────────┘
            LayerNorm               LayerNorm
                 └──────── concat (3072) ────────┘
                               |
                Fusion MLP  3072 → 512 → C   (dropout 0.3)
                               |
     Stage 1: cyclone / earthquake / flood / wildfire      (tile pairs, 224 px)
     Stage 2: no-damage / minor / major / destroyed        (building crops, 64 px)
              + damaged = minor ∪ major ∪ destroyed,  P(damaged) = P(minor)+P(major)+P(destroyed)
```

* **Real image-trained feature fusion** — both backbones and the fusion head are fine-tuned jointly with one loss. It is *not* probability averaging.
* The **existing** ConvNeXt and ViT notebooks are the baselines; they are **not** retrained here. Their numbers are loaded/embedded and compared in section 12. Probability soft-voting appears only as a clearly labelled ablation.
* Dataset pipeline, pairing, crops, preprocessing/augmentation, fold design (same seed → same fold checksums `cb0de2a1361950bf` / `e051b66afeb3dc37`), metrics, early stopping, Grad-CAM and final-test protocol are preserved from the reference notebooks.
* **Checkpoints + resume are mandatory**: every fold/stage writes `fold_k_best.pth` and `fold_k_last.pth`; re-running continues from the first unfinished fold (and mid-fold from the last epoch).

---
## 1. Experimental design (unchanged from the reference notebooks)

```
                    EXISTING DATA (hazard_data/<hazard>/{train,test})
                              |
                ┌─────────────┴─────────────┐
        train/  (~80%)                 test/  (~20%)
                |                           |  untouched until section 11
       5-fold cross-validation              |
    (Stage 1: StratifiedKFold on hazard;    |
     Stage 2: StratifiedGroupKFold on       |
     severity, grouped by source tile)      |
                └─────────────┬─────────────┘
                              ▼
         final ensemble trained on ALL development data
              → evaluated ONCE on the held-out test
```

The test split is never used for training, validation, early stopping, model selection, epoch selection or thresholds. Pre/post images of one tile are one manifest row, so they can never be separated; Stage 2 folds are grouped on the source tile so near-duplicate crops of one tile never straddle train/validation. Both properties are asserted.

**Kaggle persistence.** `/kaggle/working` survives a kernel restart inside one session, so a rerun continues from the next unfinished fold. If the *session* ends, save the notebook version (or download the results) and attach that output as an input: section 2b copies `disaster_xai_net_master_results/` from `/kaggle/input` back into `/kaggle/working` and training resumes from there.

---
## 2. Imports and configuration

In [ ]:
import os, re, gc, json, time, random, hashlib, itertools, warnings, shutil, zipfile
import datetime, platform, sys, glob, copy, importlib.util
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms
from PIL import Image
import timm
import sklearn
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, auc, matthews_corrcoef,
                             cohen_kappa_score, confusion_matrix, classification_report)
from sklearn.preprocessing import label_binarize
from sklearn.utils.class_weight import compute_class_weight
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
Image.MAX_IMAGE_PIXELS = None

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("torch", torch.__version__, "| timm", timm.__version__, "| pandas", pd.__version__,
      "| sklearn", sklearn.__version__)

### Configuration

Backbone learning rates are the ones each reference notebook used for its backbone (ConvNeXt 5e-5, ViT 2e-5); the fusion head uses the shared head LR 1e-3. ViT-B/16 + ConvNeXt-T in one graph is heavier than either alone, so the micro-batch is reduced and **gradient accumulation keeps the effective batch identical to the reference runs** (32 for Stage 1, 256 for Stage 2). Model selection / early stopping: validation **macro-F1**, `MAX_EPOCHS = 40`, `PATIENCE = 8`.

The `DXN_*` environment variables exist only for offline smoke-testing of the code; their defaults are the real protocol and nothing needs to be set on Kaggle.

In [ ]:
_env = os.environ.get

# ---------------- identity ----------------
ARCH_TAG      = "convnext_vit_fusion"
MODEL_NAME    = "ConvNeXtViTFusion"
CONV_MODEL    = _env("DXN_CONV_MODEL", "convnext_tiny")          # existing ConvNeXt notebook
VIT_MODEL     = _env("DXN_VIT_MODEL",  "vit_base_patch16_224")   # existing ViT notebook
LR_CONV       = 5e-5        # ConvNeXt LR from the ConvNeXt notebook
LR_VIT        = 2e-5        # ViT LR from the ViT notebook (ViT collapses at 1e-4)
LR_HEAD       = 1e-3        # fusion head (same as both notebooks' head LR)
FUSION_HIDDEN = 512         # same 512-unit head as the reference siamese classifier

# ---------------- protocol (must match the reference notebooks) ----------------
N_SPLITS   = 5
SEED       = 42
MAX_EPOCHS = int(_env("DXN_MAX_EPOCHS", 40))
PATIENCE   = 8

S1_IMG, S2_CROP    = 224, 64
S1_BATCH, S1_ACCUM = int(_env("DXN_S1_BATCH", 8)),   4   # 8 x 4   = effective 32
S2_BATCH, S2_ACCUM = int(_env("DXN_S2_BATCH", 128)), 2   # 128 x 2 = effective 256

MAX_PER_CLASS     = 10000
BBOX_PAD          = 0.25
DROP_UNCLASSIFIED = True

USE_PAIR     = True
DROPOUT      = 0.3
WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.05
NUM_WORKERS  = int(_env("DXN_NUM_WORKERS", 2))

# ---------------- checkpoints / resume (MANDATORY) ----------------
RESUME        = True
FORCE_RETRAIN = False
LAST_CKPT_WEIGHTS_DTYPE = "float16"   # weights-only 'last' ckpt kept after a fold completes
                                      # (full optimizer state is kept only while a fold runs)
PRETRAINED    = _env("DXN_PRETRAINED", "1") == "1"   # ImageNet weights via timm (needs internet)
SANITY_ONLY   = _env("DXN_SANITY_ONLY", "0") == "1"  # True -> stop after the sanity-check cell

EXPECTED_S1_CHECKSUM = "cb0de2a1361950bf"   # printed by both reference notebooks
EXPECTED_S2_CHECKSUM = "e051b66afeb3dc37"

# ---------------- output ----------------
OUT = _env("DXN_OUTPUT_ROOT", "/kaggle/working/disaster_xai_net_master_results")
D = {k: os.path.join(OUT, v) for k, v in {
    'ckpt':        'checkpoints',
    'ckpt_s1':     'checkpoints/stage1',
    'ckpt_s2':     'checkpoints/stage2',
    'ckpt_final':  'checkpoints/final',
    'folds':       'fold_assignments',
    's1':          'stage1_hazard',
    's2':          'stage2_severity_multiclass',
    's2b':         'stage2_severity_binary',
    'oof':         'oof_predictions',
    'metrics':     'metrics',
    'final_test':  'final_test',
    'xai':         'xai',
    'cmp':         'comparison',
    'bundle':      'deployment_bundle',
    'frontend':    'frontend',
    'repro':       'reproducibility',
}.items()}
for k, p in D.items():
    os.makedirs(p, exist_ok=True)
for s in ('s1', 's2', 's2b'):
    for f in range(1, N_SPLITS + 1):
        os.makedirs(os.path.join(D[s], f"fold_{f}"), exist_ok=True)
for st in ('stage1', 'stage2'):
    for br in ('convnext', 'vit'):
        os.makedirs(os.path.join(D['xai'], st, br), exist_ok=True)

HAZARD_NAMES = ['cyclone', 'earthquake', 'flood', 'wildfire']
DAMAGE_NAMES = ['no-damage', 'minor-damage', 'major-damage', 'destroyed']
BINARY_NAMES = ['no-damage', 'damaged']
hazard_to_idx = {n: i for i, n in enumerate(HAZARD_NAMES)}
damage_to_idx = {n: i for i, n in enumerate(DAMAGE_NAMES)}

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

set_seed(SEED)
AMP = torch.cuda.is_available()
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD  = [0.229, 0.224, 0.225]

def checksum(keys, folds):
    s = "|".join(f"{k}:{f}" for k, f in sorted(zip(map(str, keys), map(int, folds))))
    return hashlib.md5(s.encode()).hexdigest()[:16]

def jdump(obj, path):
    """atomic JSON write (safe against a disconnect mid-write)"""
    tmp = path + ".tmp"
    with open(tmp, "w") as fh:
        json.dump(obj, fh, indent=2, default=_json_default)
    os.replace(tmp, path)

def _json_default(o):
    if isinstance(o, (np.integer,)):  return int(o)
    if isinstance(o, (np.floating,)): return None if np.isnan(o) else float(o)
    if isinstance(o, np.ndarray):     return o.tolist()
    if isinstance(o, (np.bool_,)):    return bool(o)
    return str(o)

def atomic_torch_save(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)

def tload(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:                      # very old torch
        return torch.load(path, map_location="cpu")

HYPERPARAMS = dict(conv_model=CONV_MODEL, vit_model=VIT_MODEL, lr_conv=LR_CONV, lr_vit=LR_VIT,
                   lr_head=LR_HEAD, fusion_hidden=FUSION_HIDDEN, dropout=DROPOUT,
                   weight_decay=WEIGHT_DECAY, label_smoothing=LABEL_SMOOTH,
                   max_epochs=MAX_EPOCHS, patience=PATIENCE, seed=SEED, n_splits=N_SPLITS,
                   s1_img=S1_IMG, s2_crop=S2_CROP, s1_batch=S1_BATCH, s1_accum=S1_ACCUM,
                   s2_batch=S2_BATCH, s2_accum=S2_ACCUM, max_per_class=MAX_PER_CLASS,
                   bbox_pad=BBOX_PAD, drop_unclassified=DROP_UNCLASSIFIED, use_pair=USE_PAIR,
                   optimizer="AdamW", scheduler="CosineAnnealingLR(T_max=MAX_EPOCHS)",
                   loss="CrossEntropy(class-balanced weights, label smoothing)",
                   selection_metric="val_macro_f1", amp=AMP, pretrained=PRETRAINED)
jdump(HYPERPARAMS, os.path.join(D['repro'], "hyperparameters.json"))
print(f"{ARCH_TAG} | {CONV_MODEL} + {VIT_MODEL} | MAX_EPOCHS={MAX_EPOCHS} patience={PATIENCE}")
print(f"RESUME={RESUME} FORCE_RETRAIN={FORCE_RETRAIN} | results -> {OUT}")

### 2b. Restore a previous session's results (resume across Kaggle sessions)

If `checkpoints/` is empty here but an earlier run's `disaster_xai_net_master_results/` is attached under `/kaggle/input`, it is copied back so the run resumes instead of restarting. Nothing is copied if local checkpoints already exist.

In [ ]:
def _has_ckpts(root):
    return any(fn.endswith(".pth") for _, _, fs in os.walk(os.path.join(root, "checkpoints")) for fn in fs)

RESTORED_FROM = None
if RESUME and not FORCE_RETRAIN and not _has_ckpts(OUT) and os.path.isdir("/kaggle/input"):
    for dp, dn, _ in os.walk("/kaggle/input"):
        if os.path.basename(dp) == "disaster_xai_net_master_results" and _has_ckpts(dp):
            RESTORED_FROM = dp; break
    if RESTORED_FROM:
        print("restoring previous results from", RESTORED_FROM)
        shutil.copytree(RESTORED_FROM, OUT, dirs_exist_ok=True)
print("restored from:", RESTORED_FROM or "nothing (fresh run or local checkpoints present)")
_existing = sorted(os.path.relpath(os.path.join(r, f), OUT)
                   for r, _, fs in os.walk(D['ckpt']) for f in fs if f.endswith((".pth", ".json")))
print(f"{len(_existing)} checkpoint/state files already present")
for p in _existing[:30]:
    print("  ", p)

---
## 3. Load the existing split (never regenerated)

In [ ]:
hazard_base_path = _env("DXN_DATA_ROOT")
if not hazard_base_path:
    for b in ["/kaggle/input", "/content"]:
        if os.path.isdir(b):
            for dp, _, _ in os.walk(b):
                if os.path.basename(dp) == "hazard_data":
                    hazard_base_path = dp; break
        if hazard_base_path: break
assert hazard_base_path, ("hazard_data not found. On Kaggle: + Add Input > "
                          "Notebook Output > lubdham/disaster-separate-data")
print("hazard_base_path:", hazard_base_path)


def get_full_prefix(fn):
    m = re.match(r'^(.*?)_(pre|post)_disaster\.png$', fn)
    return m.group(1) if m else None

def get_event_name(prefix):
    m = re.match(r'^(.*)_\d+$', prefix)
    return m.group(1) if m else prefix

records = []
for hazard in sorted(os.listdir(hazard_base_path)):
    for split in ['train', 'test']:                 # the EXISTING split, preserved
        img_dir = os.path.join(hazard_base_path, hazard, split, 'images')
        lbl_dir = os.path.join(hazard_base_path, hazard, split, 'labels')
        if not os.path.isdir(img_dir):
            continue
        files = set(os.listdir(img_dir))
        for p in sorted({get_full_prefix(f) for f in files if get_full_prefix(f)}):
            pre_f, post_f = f"{p}_pre_disaster.png", f"{p}_post_disaster.png"
            if pre_f in files and post_f in files:  # pair kept together by construction
                records.append({
                    'hazard_type': hazard, 'orig_split': split, 'event_prefix': p,
                    'pre_path':  os.path.join(img_dir, pre_f),
                    'post_path': os.path.join(img_dir, post_f),
                    'post_label_path': os.path.join(lbl_dir, post_f.replace('.png', '.json')),
                })

all_df = (pd.DataFrame(records).sort_values('event_prefix').reset_index(drop=True)
            .sample(frac=1.0, random_state=SEED).reset_index(drop=True))
all_df['event_name'] = all_df['event_prefix'].apply(get_event_name)
all_df['label'] = all_df['hazard_type'].map(hazard_to_idx)

dev_df  = all_df[all_df.orig_split == 'train'].reset_index(drop=True)   # 5-fold CV pool
test_df = all_df[all_df.orig_split == 'test'].reset_index(drop=True)    # untouched

print(f"\ntotal pairs {len(all_df)} | development {len(dev_df)} "
      f"({len(dev_df)/len(all_df):.1%}) | held-out test {len(test_df)} "
      f"({len(test_df)/len(all_df):.1%})")
print("\nper hazard:")
print(pd.crosstab(all_df['hazard_type'], all_df['orig_split'], margins=True))
print("\nevents per hazard:")
for h, g in all_df.groupby('hazard_type'):
    print(f"  {h:11s} {g['event_name'].nunique()} event(s): {sorted(g['event_name'].unique())}")

### Leakage checks (asserted, not assumed)

In [ ]:
# 1. pre/post pairs inseparable
assert not all_df['event_prefix'].duplicated().any(), "duplicate tile prefixes"
assert all_df['pre_path'].map(os.path.exists).all(),  "missing pre image"
assert all_df['post_path'].map(os.path.exists).all(), "missing post image"
print("[OK] No pre/post leakage - every pair is one row and shares a fold by construction")

# 2. dev and test disjoint
overlap = set(dev_df['event_prefix']) & set(test_df['event_prefix'])
assert not overlap, f"{len(overlap)} tiles appear in BOTH development and test"
assert len(dev_df) + len(test_df) == len(all_df)
print("[OK] Test set disjoint from development set")
print(f"[OK] Test set untouched - {len(test_df)} pairs reserved, read only in section 11")

---
## 4. Preprocessing vs augmentation (preserved)

Training: resize 256 → random H/V flip → rotation ≤20° → mild colour jitter → random 224 crop → ImageNet normalisation. Validation/test: resize 256 → centre crop 224 → normalisation, nothing random. Stage 2 crops: the 8 dihedral transforms for training only; validation/test deterministic. Both ensemble branches receive the **same** normalised tensors (both backbones are ImageNet-normalised timm models).

In [ ]:
s1_train_tf = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomCrop(S1_IMG),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
])

s1_eval_tf = transforms.Compose([                # validation AND test - deterministic
    transforms.Resize((256, 256)),
    transforms.CenterCrop(S1_IMG),
    transforms.ToTensor(),
    transforms.Normalize(NORM_MEAN, NORM_STD),
])

_MEAN_T = torch.tensor(NORM_MEAN).view(3, 1, 1)
_STD_T  = torch.tensor(NORM_STD).view(3, 1, 1)

PREPROCESSING_CONFIG = {
    "normalization": {"mean": NORM_MEAN, "std": NORM_STD, "scale": "pixel/255"},
    "stage1": {"input": "full pre/post tile pair", "train": [
                    "Resize(256,256)", "RandomHorizontalFlip(0.5)", "RandomVerticalFlip(0.5)",
                    "RandomRotation(20)", "ColorJitter(0.2,0.2,0.2)", f"RandomCrop({S1_IMG})",
                    "ToTensor", "Normalize"],
               "eval": ["Resize(256,256)", f"CenterCrop({S1_IMG})", "ToTensor", "Normalize"],
               "note": "training tiles were first cached as 256x256 bilinear JPEG (q95)"},
    "stage2": {"input": "building crop pair cut from full-resolution pre and post tiles",
               "crop": {"bbox_from": "polygon min/max in image pixels", "pad_fraction": BBOX_PAD,
                        "min_side_px": 8, "resize": [S2_CROP, S2_CROP], "interp": "bilinear"},
               "train": ["random rot90 k in 0..3", "random hflip(0.5)", "random vflip(0.5)"],
               "eval": ["none (deterministic)"]},
}
print("train transform :", len(s1_train_tf.transforms), "ops (augmented)")
print("eval  transform :", len(s1_eval_tf.transforms), "ops (deterministic, no augmentation)")

---
## 5. Metrics (preserved + numerical exports)

Accuracy, precision/recall/F1 (macro + weighted), one-vs-rest ROC-AUC (macro + weighted), MCC, Cohen's κ, and QWK for the ordinal Stage 2 severity. Every plot is accompanied by its numbers (CSV/JSON): confusion matrices (counts and row-normalised), ROC curve points, per-class tables, histories.

In [ ]:
def compute_metrics(y, pred, prob, n_classes, ordinal=False):
    """All scalar metrics for one (fold, stage, view)."""
    m = {}
    m['accuracy'] = accuracy_score(y, pred)
    for avg in ['macro', 'weighted']:
        m[f'precision_{avg}'] = precision_score(y, pred, average=avg, zero_division=0)
        m[f'recall_{avg}']    = recall_score(y, pred, average=avg, zero_division=0)
        m[f'f1_{avg}']        = f1_score(y, pred, average=avg, zero_division=0)
    try:
        if n_classes == 2:
            m['roc_auc'] = roc_auc_score(y, prob[:, 1])
            m['roc_auc_macro'] = m['roc_auc']
            m['roc_auc_weighted'] = m['roc_auc']
        else:
            m['roc_auc_macro'] = roc_auc_score(y, prob, multi_class='ovr', average='macro',
                                               labels=list(range(n_classes)))
            m['roc_auc_weighted'] = roc_auc_score(y, prob, multi_class='ovr', average='weighted',
                                                  labels=list(range(n_classes)))
            m['roc_auc'] = m['roc_auc_macro']
    except ValueError as e:
        m['roc_auc'] = m['roc_auc_macro'] = m['roc_auc_weighted'] = np.nan
        print("   ROC-AUC unavailable:", e)
    m['mcc'] = matthews_corrcoef(y, pred)
    m['cohen_kappa'] = cohen_kappa_score(y, pred)
    if ordinal:
        m['qwk'] = cohen_kappa_score(y, pred, weights='quadratic')
    return {k: float(v) for k, v in m.items()}


def classwise_table(y, pred, names):
    """Per-class precision / recall / F1 / support + one-vs-rest accuracy."""
    rep = classification_report(y, pred, labels=list(range(len(names))),
                                target_names=names, output_dict=True, zero_division=0)
    cm = confusion_matrix(y, pred, labels=list(range(len(names))))
    total = cm.sum()
    rows = []
    for i, n in enumerate(names):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = total - tp - fn - fp
        rows.append({'class': n,
                     'precision': rep[n]['precision'], 'recall': rep[n]['recall'],
                     'f1': rep[n]['f1-score'], 'support': int(rep[n]['support']),
                     'ovr_accuracy': (tp + tn) / total})
    for key, label in [('macro avg', 'Macro Avg'), ('weighted avg', 'Weighted Avg')]:
        rows.append({'class': label, 'precision': rep[key]['precision'],
                     'recall': rep[key]['recall'], 'f1': rep[key]['f1-score'],
                     'support': int(rep[key]['support']), 'ovr_accuracy': np.nan})
    return pd.DataFrame(rows)


def plot_training_curves(hist, title, path):
    ep = range(1, len(hist['train_loss']) + 1)
    fig, ax = plt.subplots(1, 3, figsize=(17, 4))
    ax[0].plot(ep, hist['train_loss'], marker='o', ms=3, label='training loss')
    ax[0].plot(ep, hist['val_loss'],   marker='s', ms=3, label='validation loss')
    ax[0].set_ylabel('loss'); ax[0].set_title(f'{title} - loss')
    ax[1].plot(ep, hist['train_f1'], marker='o', ms=3, label='training macro-F1')
    ax[1].plot(ep, hist['val_f1'],   marker='s', ms=3, label='validation macro-F1')
    ax[1].set_ylabel('macro-F1'); ax[1].set_title(f'{title} - macro-F1')
    ax[2].plot(ep, hist['train_acc'], marker='o', ms=3, label='training accuracy')
    ax[2].plot(ep, hist['val_acc'],   marker='s', ms=3, label='validation accuracy')
    ax[2].set_ylabel('accuracy'); ax[2].set_title(f'{title} - accuracy')
    for a in ax:
        a.set_xlabel('epoch'); a.legend(); a.grid(alpha=.3)
        if hist.get('best_epoch'):
            a.axvline(hist['best_epoch'], ls='--', c='k', lw=1, alpha=.6)
    plt.tight_layout(); plt.savefig(path, dpi=150, bbox_inches='tight'); plt.show(); plt.close(fig)


def plot_confusion(y, pred, names, title, path, csv_prefix=None):
    cm = confusion_matrix(y, pred, labels=list(range(len(names))))
    cmn = cm.astype(float) / np.maximum(cm.sum(1, keepdims=True), 1)
    fig, ax = plt.subplots(1, 2, figsize=(6 + 2.2 * len(names), 4.4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax[0],
                xticklabels=names, yticklabels=names, cbar=False)
    ax[0].set_xlabel('Predicted'); ax[0].set_ylabel('Actual'); ax[0].set_title(f'{title} - counts')
    sns.heatmap(cmn, annot=True, fmt='.3f', cmap='Blues', ax=ax[1], vmin=0, vmax=1,
                xticklabels=names, yticklabels=names, cbar=False)
    ax[1].set_xlabel('Predicted'); ax[1].set_ylabel('Actual')
    ax[1].set_title(f'{title} - row-normalised (recall)')
    plt.tight_layout(); plt.savefig(path, dpi=150, bbox_inches='tight'); plt.show(); plt.close(fig)
    if csv_prefix:
        pd.DataFrame(cm, index=names, columns=names).to_csv(csv_prefix + ".csv")
        pd.DataFrame(cmn, index=names, columns=names).to_csv(csv_prefix + "_normalized.csv")
    return cm


def plot_roc(y, prob, names, title, path, points_csv=None):
    """One-vs-rest ROC with micro and macro averages. Returns dict of AUCs."""
    n = len(names)
    aucs, pts = {}, []
    fig = plt.figure(figsize=(6.8, 5.4))
    if n == 2:
        fpr, tpr, _ = roc_curve(y, prob[:, 1])
        aucs['damaged'] = auc(fpr, tpr)
        pts += [{'curve': 'damaged', 'fpr': a, 'tpr': b} for a, b in zip(fpr, tpr)]
        plt.plot(fpr, tpr, lw=2, label=f"damaged (AUC {aucs['damaged']:.4f})")
    else:
        Y = label_binarize(y, classes=list(range(n)))
        fprs, tprs = {}, {}
        for i, nm in enumerate(names):
            if Y[:, i].sum() == 0 or Y[:, i].sum() == len(Y):
                continue
            fprs[i], tprs[i], _ = roc_curve(Y[:, i], prob[:, i])
            aucs[nm] = auc(fprs[i], tprs[i])
            pts += [{'curve': nm, 'fpr': a, 'tpr': b} for a, b in zip(fprs[i], tprs[i])]
            plt.plot(fprs[i], tprs[i], lw=1.6, label=f"{nm} (AUC {aucs[nm]:.4f})")
        fpr_mi, tpr_mi, _ = roc_curve(Y.ravel(), prob.ravel())
        aucs['micro'] = auc(fpr_mi, tpr_mi)
        pts += [{'curve': 'micro', 'fpr': a, 'tpr': b} for a, b in zip(fpr_mi, tpr_mi)]
        plt.plot(fpr_mi, tpr_mi, ls=':', lw=2.4, color='deeppink',
                 label=f"micro-average (AUC {aucs['micro']:.4f})")
        if fprs:
            grid = np.unique(np.concatenate([fprs[i] for i in fprs]))
            mean_tpr = np.zeros_like(grid)
            for i in fprs:
                mean_tpr += np.interp(grid, fprs[i], tprs[i])
            mean_tpr /= len(fprs)
            aucs['macro'] = auc(grid, mean_tpr)
            pts += [{'curve': 'macro', 'fpr': a, 'tpr': b} for a, b in zip(grid, mean_tpr)]
            plt.plot(grid, mean_tpr, ls=':', lw=2.4, color='navy',
                     label=f"macro-average (AUC {aucs['macro']:.4f})")
    plt.plot([0, 1], [0, 1], 'k--', lw=1, alpha=.5)
    plt.xlim(0, 1); plt.ylim(0, 1.02)
    plt.xlabel('False positive rate'); plt.ylabel('True positive rate')
    plt.title(title); plt.legend(loc='lower right', fontsize=8); plt.grid(alpha=.3)
    plt.tight_layout(); plt.savefig(path, dpi=150, bbox_inches='tight'); plt.show(); plt.close(fig)
    if points_csv:
        pd.DataFrame(pts).to_csv(points_csv, index=False)
    return {k: float(v) for k, v in aucs.items()}


def summary_table(fold_metrics, keys):
    """metric x fold1..fold5 + mean + std"""
    rows = []
    for k in keys:
        vals = [fm.get(k, np.nan) for fm in fold_metrics]
        row = {'Metric': k}
        row.update({f'Fold {i+1}': v for i, v in enumerate(vals)})
        row['Mean'] = np.nanmean(vals); row['Std'] = np.nanstd(vals, ddof=1) if len(vals) > 1 else 0.0
        rows.append(row)
    return pd.DataFrame(rows)


def to_binary(y, prob):
    """damaged = minor + major + destroyed;  P(damaged) = P(minor)+P(major)+P(destroyed)."""
    yb = (np.asarray(y) > 0).astype(int)
    p_dmg = prob[:, 1] + prob[:, 2] + prob[:, 3]
    pb = np.stack([prob[:, 0], p_dmg], 1)
    return yb, (p_dmg > 0.5).astype(int), pb


def full_report(y, pred, prob, names, title, d, prefix="", ordinal=False, hist=None, extra=None):
    """Writes every per-view artifact into directory d; returns the metric dict."""
    n = len(names)
    m = compute_metrics(y, pred, prob, n, ordinal=ordinal)
    if extra: m.update(extra)
    print("\n".join(f"  {k:22s} {v:.4f}" for k, v in m.items() if isinstance(v, float)))
    rep_txt = classification_report(y, pred, labels=list(range(n)), target_names=names,
                                    digits=4, zero_division=0)
    print("\nclassification report\n" + rep_txt)
    with open(os.path.join(d, f"{prefix}classification_report.txt"), "w") as fh:
        fh.write(rep_txt)
    jdump(classification_report(y, pred, labels=list(range(n)), target_names=names,
                                output_dict=True, zero_division=0),
          os.path.join(d, f"{prefix}classification_report.json"))
    cw = classwise_table(y, pred, names)
    cw.to_csv(os.path.join(d, f"{prefix}classwise_metrics.csv"), index=False)
    if hist is not None:
        plot_training_curves(hist, title, os.path.join(d, f"{prefix}training_curves.png"))
        pd.DataFrame({k: v for k, v in hist.items() if isinstance(v, list)}).to_csv(
            os.path.join(d, f"{prefix}training_history.csv"), index=False)
    plot_confusion(y, pred, names, title, os.path.join(d, f"{prefix}confusion_matrix.png"),
                   csv_prefix=os.path.join(d, f"{prefix}confusion_matrix"))
    aucs = plot_roc(y, prob, names, f"{title} ROC", os.path.join(d, f"{prefix}roc_curves.png"),
                    points_csv=os.path.join(d, f"{prefix}roc_curve_points.csv"))
    pd.Series(aucs).to_frame('auc').to_csv(os.path.join(d, f"{prefix}roc_auc_values.csv"))
    pd.DataFrame([m]).to_csv(os.path.join(d, f"{prefix}metrics.csv"), index=False)
    jdump(m, os.path.join(d, f"{prefix}metrics.json"))
    return m

print("metric helpers ready")

---
## 6. The trained ConvNeXt + ViT fusion ensemble, checkpointing and the resumable CV driver

**Branches.** Each branch is the reference *siamese* design: one shared-weight timm backbone runs over the pre image and over the post image, and the two pooled feature vectors are concatenated (`f_pre ⊕ f_post`). The post image is passed first in each branch, which the Grad-CAM hooks rely on.

**Fusion.** Each branch vector gets its own LayerNorm (so neither branch dominates the fused vector by feature scale), the two are concatenated, and a fusion MLP with the reference head shape (dropout → 512 → ReLU → dropout → C) produces the logits. A single cross-entropy loss trains **both backbones and the head jointly**; backbone LRs are the ones from each reference notebook.

**Checkpoint layout.**
```
checkpoints/stage{1,2}/fold_k_best.pth         best-val-macro-F1 weights + metadata
checkpoints/stage{1,2}/fold_k_last.pth         while running: full resumable state (model/opt/sched/scaler/RNG/history)
                                               after the fold: last-epoch weights + metadata (optimizer dropped)
checkpoints/stage{1,2}/fold_k_best_val_outputs.npz   validation preds/probs of the best epoch
checkpoints/stage{1,2}/fold_k_done.json        completion marker (fold row + history)
checkpoints/final/stage{1,2}_ensemble_final.pth      final models trained on the full dev set
```
All writes are atomic (`tmp` → `os.replace`), so a disconnect mid-save cannot corrupt a checkpoint. On rerun: completed folds are loaded and skipped; an interrupted fold resumes from its last epoch.

In [ ]:
class SiameseBranch(nn.Module):
    """Reference siamese design: one shared-weight backbone over (pre, post) -> concat."""
    def __init__(self, model_name, img_size=None, pretrained=True):
        super().__init__()
        kw = dict(pretrained=pretrained, num_classes=0)
        if img_size is not None and any(t in model_name for t in ('vit', 'deit', 'beit')):
            kw['img_size'] = img_size          # CNNs accept any input size as-is
        self.backbone = timm.create_model(model_name, **kw)
        self.img_feat_dim = self.backbone.num_features
        self.feat_dim = self.img_feat_dim * (2 if USE_PAIR else 1)

    def forward(self, pre_img, post_img):
        f_post = self.backbone(post_img)        # post first - Grad-CAM relies on this
        if USE_PAIR:
            f_pre = self.backbone(pre_img)      # SAME weights -> siamese
            return torch.cat([f_pre, f_post], dim=1)
        return f_post


class ConvNeXtViTFusion(nn.Module):
    """Trained feature-fusion ensemble: ConvNeXt siamese ⊕ ViT siamese -> fusion MLP."""
    def __init__(self, num_classes, vit_img_size=None, pretrained=True,
                 dropout=DROPOUT, hidden=FUSION_HIDDEN):
        super().__init__()
        self.conv = SiameseBranch(CONV_MODEL, None, pretrained)
        self.vit  = SiameseBranch(VIT_MODEL, vit_img_size, pretrained)
        self.conv_norm = nn.LayerNorm(self.conv.feat_dim)
        self.vit_norm  = nn.LayerNorm(self.vit.feat_dim)
        self.fused_dim = self.conv.feat_dim + self.vit.feat_dim
        self.fusion = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(self.fused_dim, hidden), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(hidden, num_classes))

    def branch_features(self, pre_img, post_img):
        return self.conv(pre_img, post_img), self.vit(pre_img, post_img)

    def forward(self, pre_img, post_img, return_features=False):
        fc, fv = self.branch_features(pre_img, post_img)
        z = torch.cat([self.conv_norm(fc), self.vit_norm(fv)], dim=1)
        out = self.fusion(z)
        if return_features:
            return out, {'convnext': fc, 'vit': fv, 'fused': z}
        return out


def feature_dims(model):
    return {'convnext_per_image': model.conv.img_feat_dim, 'convnext_branch': model.conv.feat_dim,
            'vit_per_image': model.vit.img_feat_dim, 'vit_branch': model.vit.feat_dim,
            'fused': model.fused_dim, 'fusion_hidden': FUSION_HIDDEN}


def param_groups(model):
    head = [p for n, p in model.named_parameters()
            if not (n.startswith('conv.backbone.') or n.startswith('vit.backbone.'))]
    return [{'params': list(model.conv.backbone.parameters()), 'lr': LR_CONV, 'name': 'convnext_backbone'},
            {'params': list(model.vit.backbone.parameters()),  'lr': LR_VIT,  'name': 'vit_backbone'},
            {'params': head,                                   'lr': LR_HEAD, 'name': 'fusion_head'}]


STAGE_SPEC = {
    'stage1': dict(task='hazard', class_names=HAZARD_NAMES, vit_img_size=None,
                   input_size={'convnext': S1_IMG, 'vit': S1_IMG}, input_kind='tile_pair'),
    'stage2': dict(task='damage', class_names=DAMAGE_NAMES, vit_img_size=S2_CROP,
                   input_size={'convnext': S2_CROP, 'vit': S2_CROP}, input_kind='building_crop_pair'),
}
FOLD_CHECKSUMS = {}      # filled in sections 7/8
FEATURE_DIMS = {}

def make_ensemble(stage, pretrained=None):
    spec = STAGE_SPEC[stage]
    return ConvNeXtViTFusion(len(spec['class_names']), vit_img_size=spec['vit_img_size'],
                             pretrained=PRETRAINED if pretrained is None else pretrained)


def ckpt_meta(stage, fold, epoch, best_metric, kind, extra=None):
    spec = STAGE_SPEC[stage]
    meta = {
        'architecture': {'name': MODEL_NAME, 'arch_tag': ARCH_TAG, 'type': 'trained feature-fusion ensemble',
                         'conv_model': CONV_MODEL, 'vit_model': VIT_MODEL, 'siamese': True,
                         'use_pair': USE_PAIR, 'shared_weights_pre_post': True,
                         'branch_norm': 'LayerNorm', 'fusion': 'concat -> MLP',
                         'fusion_hidden': FUSION_HIDDEN, 'dropout': DROPOUT,
                         'vit_img_size': spec['vit_img_size'], 'num_classes': len(spec['class_names'])},
        'stage': stage, 'task': spec['task'], 'fold': fold, 'epoch': epoch,
        'checkpoint_kind': kind,
        'best_metric': {'name': 'val_macro_f1', 'value': best_metric},
        'class_mapping': {str(i): n for i, n in enumerate(spec['class_names'])},
        'input_size': spec['input_size'], 'input_kind': spec['input_kind'],
        'normalization': {'mean': NORM_MEAN, 'std': NORM_STD},
        'hyperparameters': HYPERPARAMS,
        'feature_dimensions': FEATURE_DIMS.get(stage),
        'fold_checksum': FOLD_CHECKSUMS.get(stage),
        'seed': SEED, 'n_splits': N_SPLITS,
        'torch_version': torch.__version__, 'timm_version': timm.__version__,
        'created': datetime.datetime.now().isoformat(timespec='seconds'),
    }
    if extra: meta.update(extra)
    return meta


def cpu_state(model, dtype=None):
    sd = {}
    for k, v in model.state_dict().items():
        v = v.detach().cpu()
        if dtype is not None and v.is_floating_point():
            v = v.to(dtype)
        sd[k] = v.clone()
    return sd


def load_weights(model, ckpt_path):
    ck = tload(ckpt_path)
    model.load_state_dict({k: (v.float() if v.is_floating_point() else v)
                           for k, v in ck['state_dict'].items()})
    return ck.get('meta', {})


def capture_rng():
    return {'py': random.getstate(), 'np': np.random.get_state(), 'torch': torch.get_rng_state(),
            'cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}

def restore_rng(s):
    random.setstate(s['py']); np.random.set_state(s['np']); torch.set_rng_state(s['torch'])
    if s.get('cuda') is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(s['cuda'])


def run_epoch(model, loader, criterion, optimizer=None, scaler=None, desc="", accum=1):
    training = optimizer is not None
    model.train() if training else model.eval()
    tot, n, P, Y, PR = 0.0, 0, [], [], []
    if training:
        optimizer.zero_grad(set_to_none=True)
    with torch.set_grad_enabled(training):
        for step, (pre, post, y) in enumerate(tqdm(loader, desc=desc, leave=False)):
            post = post.to(device, non_blocking=True)
            pre = pre.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            with torch.amp.autocast('cuda', enabled=AMP):
                out = model(pre, post)
                loss = criterion(out, y)
            if training:
                scaler.scale(loss / accum).backward()
                if (step + 1) % accum == 0:
                    scaler.step(optimizer); scaler.update()
                    optimizer.zero_grad(set_to_none=True)
            b = y.size(0); tot += loss.item() * b; n += b
            PR.append(F.softmax(out.float(), 1).detach().cpu().numpy())
            P.append(out.argmax(1).detach().cpu().numpy())
            Y.append(y.detach().cpu().numpy())
    P, Y, PR = np.concatenate(P), np.concatenate(Y), np.concatenate(PR)
    return (tot / n, accuracy_score(Y, P), f1_score(Y, P, average='macro', zero_division=0),
            P, Y, PR)


class SimulatedInterruption(RuntimeError):
    """raised only by the sanity check to test resume"""


def fold_paths(ckpt_dir, fold):
    k = fold + 1
    return {'best':    os.path.join(ckpt_dir, f"fold_{k}_best.pth"),
            'last':    os.path.join(ckpt_dir, f"fold_{k}_last.pth"),
            'bestval': os.path.join(ckpt_dir, f"fold_{k}_best_val_outputs.npz"),
            'done':    os.path.join(ckpt_dir, f"fold_{k}_done.json")}


def run_cv(stage, n_items, labels, make_datasets, batch_size, accum=1,
           ckpt_dir=None, folds=None, max_epochs=None, make_model=None, interrupt_after=None,
           verbose=True):
    """Resumable 5-fold CV for the fusion ensemble.
    Returns per-fold frame, OOF predictions, OOF probabilities, per-fold histories."""
    n_classes = len(STAGE_SPEC[stage]['class_names'])
    ckpt_dir = ckpt_dir or D['ckpt_s1' if stage == 'stage1' else 'ckpt_s2']
    folds = list(range(N_SPLITS)) if folds is None else folds
    max_epochs = max_epochs or MAX_EPOCHS
    make_model = make_model or (lambda: make_ensemble(stage))
    os.makedirs(ckpt_dir, exist_ok=True)

    oof_pred = np.full(n_items, -1, dtype=int)
    oof_prob = np.zeros((n_items, n_classes), dtype=np.float32)
    rows, hists = [], {}
    t_all = time.time()

    for fold in folds:
        P_ = fold_paths(ckpt_dir, fold)
        print("\n" + "=" * 64)
        print(f"[{stage}] FOLD {fold + 1} / {N_SPLITS}")
        print("=" * 64)
        tr_ds, va_ds, va_idx = make_datasets(fold)

        if FORCE_RETRAIN:
            for p in P_.values():
                if os.path.exists(p): os.remove(p)

        # ---------- completed fold -> load and skip ----------
        if RESUME and all(os.path.exists(P_[k]) for k in ('done', 'best', 'bestval')):
            done = json.load(open(P_['done']))
            bv = np.load(P_['bestval'])
            assert np.array_equal(bv['va_idx'], va_idx), "fold assignment changed since checkpointing"
            oof_pred[va_idx] = bv['pred']; oof_prob[va_idx] = bv['prob']
            hists[fold] = done['history']; rows.append(done['row'])
            print(f"  [RESUME] fold {fold+1} already complete - loaded "
                  f"(best epoch {done['row']['best_epoch']}, val macro-F1 "
                  f"{done['row']['best_val_macro_f1']:.4f}); not retrained")
            continue

        set_seed(SEED + fold)
        tr_ld = DataLoader(tr_ds, batch_size=batch_size, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True, drop_last=len(tr_ds) > batch_size)
        va_ld = DataLoader(va_ds, batch_size=batch_size * 2, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)
        print(f"train {len(tr_ds)} (augmented) | validation {len(va_ds)} (no augmentation)")

        model = make_model().to(device)            # FRESH pretrained weights every fold
        w = compute_class_weight('balanced', classes=np.arange(n_classes),
                                 y=labels[np.setdiff1d(np.arange(n_items), va_idx)])
        criterion = nn.CrossEntropyLoss(
            weight=torch.tensor(w, dtype=torch.float32, device=device),
            label_smoothing=LABEL_SMOOTH)
        opt = torch.optim.AdamW(param_groups(model), weight_decay=WEIGHT_DECAY)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCHS)
        scaler = torch.amp.GradScaler('cuda', enabled=AMP)

        h = {'train_loss': [], 'train_acc': [], 'train_f1': [],
             'val_loss': [], 'val_acc': [], 'val_f1': [], 'lr_conv': [], 'lr_vit': [], 'lr_head': []}
        best_f1, best_ep, bad, stopped, start = -1.0, 0, 0, False, 1
        best_pred = best_prob = None

        # ---------- interrupted fold -> resume from last epoch ----------
        if RESUME and os.path.exists(P_['last']):
            ck = tload(P_['last'])
            if ck.get('resumable'):
                model.load_state_dict(ck['state_dict'])
                opt.load_state_dict(ck['optimizer']); sched.load_state_dict(ck['scheduler'])
                scaler.load_state_dict(ck['scaler'])
                h, best_f1, best_ep = ck['history'], ck['best_f1'], ck['best_epoch']
                bad, stopped, start = ck['bad'], ck['stopped'], ck['epoch'] + 1
                restore_rng(ck['rng'])
                if os.path.exists(P_['bestval']):
                    bv = np.load(P_['bestval']); best_pred, best_prob = bv['pred'], bv['prob']
                print(f"  [RESUME] continuing fold {fold+1} from epoch {start} "
                      f"(best so far {best_f1:.4f} @ epoch {best_ep}, patience counter {bad})")
            del ck

        ep = start - 1
        if not stopped:
            for ep in range(start, max_epochs + 1):
                t0 = time.time()
                trl, tra, trf, _, _, _ = run_epoch(model, tr_ld, criterion, opt, scaler,
                                                   desc=f"F{fold+1} e{ep} train", accum=accum)
                vl, va, vf, vp, _, vpr = run_epoch(model, va_ld, criterion,
                                                   desc=f"F{fold+1} e{ep} val")
                lrs = [g['lr'] for g in opt.param_groups]
                sched.step()
                for k_, v_ in zip(['train_loss', 'train_acc', 'train_f1', 'val_loss', 'val_acc',
                                   'val_f1', 'lr_conv', 'lr_vit', 'lr_head'],
                                  [trl, tra, trf, vl, va, vf] + lrs):
                    h[k_].append(float(v_))

                mark = ""
                if vf > best_f1:
                    best_f1, best_ep, bad = vf, ep, 0
                    best_pred, best_prob = vp, vpr
                    atomic_torch_save({'state_dict': cpu_state(model),
                                       'meta': ckpt_meta(stage, fold + 1, ep, float(vf), 'best')},
                                      P_['best'])
                    np.savez(P_['bestval'] + ".tmp.npz", pred=vp, prob=vpr, va_idx=va_idx)
                    os.replace(P_['bestval'] + ".tmp.npz", P_['bestval'])
                    mark = "  <- best (saved)"
                else:
                    bad += 1
                stopped = bad >= PATIENCE
                atomic_torch_save({'resumable': True, 'epoch': ep, 'state_dict': model.state_dict(),
                                   'optimizer': opt.state_dict(), 'scheduler': sched.state_dict(),
                                   'scaler': scaler.state_dict(), 'history': h, 'best_f1': best_f1,
                                   'best_epoch': best_ep, 'bad': bad, 'stopped': stopped,
                                   'rng': capture_rng(),
                                   'meta': ckpt_meta(stage, fold + 1, ep, float(best_f1), 'last_resumable')},
                                  P_['last'])
                if verbose:
                    print(f"  epoch {ep:2d}/{max_epochs} | train loss {trl:.4f} acc {tra:.4f} F1 {trf:.4f} "
                          f"| val loss {vl:.4f} acc {va:.4f} macroF1 {vf:.4f} "
                          f"| {time.time()-t0:.0f}s{mark}")
                if interrupt_after is not None and ep == interrupt_after:
                    raise SimulatedInterruption(f"simulated disconnect after epoch {ep}")
                if stopped:
                    print(f"  EARLY STOPPING at epoch {ep} "
                          f"(no validation macro-F1 gain for {PATIENCE} epochs)")
                    break

        epochs_run = len(h['val_f1'])
        h['best_epoch'], h['epochs_run'], h['early_stopped'] = best_ep, epochs_run, bool(stopped)
        hists[fold] = h
        oof_pred[va_idx] = best_pred
        oof_prob[va_idx] = best_prob

        # slim the 'last' checkpoint: last-epoch weights + metadata, no optimizer state
        atomic_torch_save({'resumable': False,
                           'state_dict': cpu_state(model, getattr(torch, LAST_CKPT_WEIGHTS_DTYPE)),
                           'weights_dtype': LAST_CKPT_WEIGHTS_DTYPE,
                           'meta': ckpt_meta(stage, fold + 1, epochs_run, float(best_f1), 'last',
                                             extra={'history': h})}, P_['last'])
        row = {'fold': fold + 1, 'n_train': len(tr_ds), 'n_val': len(va_ds),
               'best_epoch': best_ep, 'epochs_run': epochs_run, 'early_stopped': bool(stopped),
               'best_val_macro_f1': float(best_f1)}
        rows.append(row)
        jdump({'row': row, 'history': h, 'stage': stage}, P_['done'])
        print(f"  --> fold {fold+1}: best epoch {best_ep}, {epochs_run} epochs run, "
              f"early stopping {'YES' if stopped else 'NO'}, val macro-F1 {best_f1:.4f}")

        del model, opt, sched, scaler, tr_ld, va_ld, tr_ds, va_ds
        gc.collect(); torch.cuda.empty_cache()

    print(f"\n[{stage}] complete in {(time.time()-t_all)/60:.1f} min")
    return pd.DataFrame(rows), oof_pred, oof_prob, hists

print("fusion model, checkpointing and resumable CV driver ready")

### Datasets (preserved) and Stage 2 polygon parsing

`parse_polygons` is the reference parser, extended only to also keep the polygon itself, the building `uid`, and the `lng_lat` polygon **when the label file contains it** (needed by the frontend). The filtering and enumeration order are unchanged, so every `crop_id` — and therefore the Stage 2 fold checksum — is identical to the reference notebooks. Latitude/longitude are never invented: if a label has no `lng_lat` entry the field is `null`.

In [ ]:
class TilePairDataset(Dataset):
    def __init__(self, df, tf, pre_col='pre_img', post_col='post_img'):
        self.df, self.tf = df.reset_index(drop=True), tf
        self.pre_col, self.post_col = pre_col, post_col
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        post = self.tf(Image.open(r[self.post_col]).convert('RGB'))
        pre = self.tf(Image.open(r[self.pre_col]).convert('RGB'))
        return pre, post, torch.tensor(r['label'], dtype=torch.long)


def _wkt_points(wkt):
    m = re.search(r'\(\((.*?)\)\)', wkt or '')
    if not m:
        return None
    try:
        return np.array([[float(v) for v in c.strip().split()[:2]] for c in m.group(1).split(',')])
    except ValueError:
        return None


def parse_polygons(label_path):
    with open(label_path) as fh:
        data = json.load(fh)
    feats = data.get('features', {})
    lnglat = {}
    for feat in feats.get('lng_lat', []) or []:
        uid = feat.get('properties', {}).get('uid')
        pts = _wkt_points(feat.get('wkt', ''))
        if uid is not None and pts is not None and len(pts) >= 3:
            lnglat[uid] = (feat.get('wkt'), float(pts[:, 0].mean()), float(pts[:, 1].mean()))
    out = []
    for feat in feats.get('xy', []):
        props = feat.get('properties', {})
        sub = props.get('subtype', 'un-classified')
        wkt = feat.get('wkt', '')
        m = re.search(r'\(\((.*?)\)\)', wkt)
        if not m:
            continue
        pts = np.array([[float(v) for v in c.strip().split()]
                        for c in m.group(1).split(',')])
        if len(pts) < 3:
            continue
        x0, y0 = pts[:, :2].min(0); x1, y1 = pts[:, :2].max(0)
        uid = props.get('uid')
        ll = lnglat.get(uid)
        out.append({'subtype': sub, 'x0': x0, 'y0': y0, 'x1': x1, 'y1': y1,
                    'uid': uid, 'polygon_wkt_xy': wkt,
                    'lnglat_wkt': ll[0] if ll else None,
                    'lon_centroid': ll[1] if ll else None, 'lat_centroid': ll[2] if ll else None})
    return out


def build_polys(df):
    rows = []
    for r in tqdm(df.itertuples(), total=len(df), desc="parsing labels"):
        for k, p in enumerate(parse_polygons(r.post_label_path)):
            rows.append({'crop_id': f"{r.event_prefix}#{k}", 'event_prefix': r.event_prefix,
                         'hazard_type': r.hazard_type, 'event_name': r.event_name,
                         'pre_path': r.pre_path, 'post_path': r.post_path, **p})
    return pd.DataFrame(rows)


def padded_box(x0, y0, x1, y1, W, H):
    w, h = x1 - x0, y1 - y0
    px, py = w * BBOX_PAD, h * BBOX_PAD
    bx0, by0 = max(0, x0 - px), max(0, y0 - py)
    bx1, by1 = min(W, x1 + px), min(H, y1 + py)
    if bx1 - bx0 < 8:  bx0, bx1 = max(0, bx0 - 4), min(W, bx1 + 4)
    if by1 - by0 < 8:  by0, by1 = max(0, by0 - 4), min(H, by1 + 4)
    return (int(bx0), int(by0), int(max(bx0 + 1, bx1)), int(max(by0 + 1, by1)))


class CropDataset(Dataset):
    """train=True applies the 8 dihedral augmentations; train=False is deterministic."""
    def __init__(self, idxs, train, pre_arr=None, post_arr=None, labels=None):
        self.idxs = np.asarray(idxs); self.train = train
        self.pre = arr_pre if pre_arr is None else pre_arr
        self.post = arr_post if post_arr is None else post_arr
        self.labels = sel['label'].values if labels is None else labels
    def __len__(self):
        return len(self.idxs)
    def _prep(self, a):
        t = torch.from_numpy(np.ascontiguousarray(a)).permute(2, 0, 1).float().div_(255)
        return (t - _MEAN_T) / _STD_T
    def __getitem__(self, i):
        j = self.idxs[i]
        post, pre = self.post[j], self.pre[j]
        if self.train:                                   # AUGMENTATION - training only
            k = random.randint(0, 3)
            if k:
                post, pre = np.rot90(post, k), np.rot90(pre, k)
            if random.random() < 0.5:
                post, pre = post[:, ::-1], pre[:, ::-1]
            if random.random() < 0.5:
                post, pre = post[::-1], pre[::-1]
        return self._prep(pre), self._prep(post), torch.tensor(self.labels[j], dtype=torch.long)

print("datasets ready")

---
## 6b. Lightweight sanity check (no real training)

Runs in a couple of minutes on a GPU and touches every moving part before the expensive run: dataset discovery, one batch, ConvNeXt branch forward, ViT branch forward, fusion forward, one backward pass (gradients reach both backbones and the head), checkpoint save → load round-trip, the **resume logic of the real `run_cv`** (a simulated disconnect after epoch 1, a resume that continues at epoch 2, and a third call that skips the completed fold), output directories, and ZIP creation/verification. The resume test uses a handful of building crops with cyclic placeholder labels purely to exercise the code path. Everything it writes goes to `_sanity/`, which is deleted afterwards (only the report is kept).

Set `SANITY_ONLY = True` in the config cell to stop here.

In [ ]:
SANITY_DIR = os.path.join(OUT, "_sanity")
shutil.rmtree(SANITY_DIR, ignore_errors=True); os.makedirs(SANITY_DIR, exist_ok=True)
sanity = {'started': datetime.datetime.now().isoformat(timespec='seconds')}

# 1. dataset discovery
assert len(dev_df) > 0 and len(test_df) > 0
sanity['dataset'] = {'root': hazard_base_path, 'dev_pairs': len(dev_df), 'test_pairs': len(test_df)}
print(f"[1] dataset discovery OK: {len(dev_df)} dev / {len(test_df)} test pairs")

# 2. one batch (raw paths, training transform)
_b = dev_df.groupby('hazard_type', group_keys=False).head(1).head(2)
_ld = DataLoader(TilePairDataset(_b, s1_train_tf, 'pre_path', 'post_path'), batch_size=len(_b))
pre_b, post_b, y_b = next(iter(_ld))
assert pre_b.shape == post_b.shape == (len(_b), 3, S1_IMG, S1_IMG)
print(f"[2] one batch OK: pre {tuple(pre_b.shape)} post {tuple(post_b.shape)} labels {y_b.tolist()}")

# 3-6. forward passes + one backward pass (Stage 1 geometry)
_m = make_ensemble('stage1').to(device)
FEATURE_DIMS['stage1'] = feature_dims(_m)
pre_b, post_b, y_b = pre_b.to(device), post_b.to(device), y_b.to(device)
with torch.amp.autocast('cuda', enabled=AMP):
    fc = _m.conv(pre_b, post_b);  print(f"[3] ConvNeXt siamese branch forward OK: {tuple(fc.shape)}")
    fv = _m.vit(pre_b, post_b);   print(f"[4] ViT siamese branch forward OK:      {tuple(fv.shape)}")
    out, feats = _m(pre_b, post_b, return_features=True)
    print(f"[5] fusion forward OK: fused {tuple(feats['fused'].shape)} -> logits {tuple(out.shape)}")
    loss = F.cross_entropy(out.float(), y_b)
loss.backward()
g = {n: sum(float(p.grad.abs().sum()) for p in mod.parameters() if p.grad is not None)
     for n, mod in [('convnext_backbone', _m.conv.backbone), ('vit_backbone', _m.vit.backbone),
                    ('fusion_head', _m.fusion)]}
assert all(v > 0 for v in g.values()), f"a component received no gradient: {g}"
print(f"[6] backward OK, gradient mass: " + ", ".join(f"{k}={v:.3e}" for k, v in g.items()))
sanity['stage1_shapes'] = {'conv': list(fc.shape), 'vit': list(fv.shape), 'fused': list(feats['fused'].shape),
                           'logits': list(out.shape), 'grad_mass': g}
n_params = {'convnext': sum(p.numel() for p in _m.conv.parameters()) / 1e6,
            'vit': sum(p.numel() for p in _m.vit.parameters()) / 1e6,
            'total': sum(p.numel() for p in _m.parameters()) / 1e6}
S1_PARAMS_M = round(n_params['total'], 1)
print(f"    params (M): {', '.join(f'{k} {v:.1f}' for k, v in n_params.items())}")

# 7. checkpoint save / load round trip
_m.eval()
with torch.no_grad():
    ref = _m(pre_b, post_b).float()
ck_path = os.path.join(SANITY_DIR, "roundtrip.pth")
atomic_torch_save({'state_dict': cpu_state(_m), 'meta': ckpt_meta('stage1', 0, 0, None, 'sanity')}, ck_path)
del _m; gc.collect(); torch.cuda.empty_cache()
_m2 = make_ensemble('stage1', pretrained=False).to(device).eval()
meta_rt = load_weights(_m2, ck_path)
with torch.no_grad():
    rt = _m2(pre_b, post_b).float()
assert torch.allclose(ref, rt, atol=1e-4), "checkpoint round-trip changed the outputs"
assert meta_rt['class_mapping']['0'] == HAZARD_NAMES[0] and meta_rt['feature_dimensions']
print(f"[7] checkpoint save/load OK ({os.path.getsize(ck_path)/1e6:.0f} MB), "
      f"max |diff| {float((ref-rt).abs().max()):.2e}; metadata keys: {sorted(meta_rt)[:6]}...")
del _m2, pre_b, post_b; gc.collect(); torch.cuda.empty_cache()

# 8. Stage 2 crop path + forward
_poly = None
for _r in dev_df.itertuples():
    _pp = parse_polygons(_r.post_label_path)
    if len(_pp) >= 4:
        _poly = (_r, _pp[:8]); break
assert _poly is not None, "no label file with >=4 building polygons found"
_r, _pp = _poly
_po, _pr = Image.open(_r.post_path).convert('RGB'), Image.open(_r.pre_path).convert('RGB')
_W, _H = _po.size
_cp = np.stack([np.asarray(_pr.crop(padded_box(p['x0'], p['y0'], p['x1'], p['y1'], _W, _H))
                           .resize((S2_CROP, S2_CROP), Image.BILINEAR)) for p in _pp])
_cq = np.stack([np.asarray(_po.crop(padded_box(p['x0'], p['y0'], p['x1'], p['y1'], _W, _H))
                           .resize((S2_CROP, S2_CROP), Image.BILINEAR)) for p in _pp])
_lab = np.arange(len(_pp)) % 4                          # placeholder labels (code path only)
_cds = CropDataset(np.arange(len(_pp)), True, pre_arr=_cp, post_arr=_cq, labels=_lab)
_m = make_ensemble('stage2').to(device).eval()
FEATURE_DIMS['stage2'] = feature_dims(_m)
_x = next(iter(DataLoader(_cds, batch_size=len(_cds))))
with torch.no_grad():
    o2 = _m(_x[0].to(device), _x[1].to(device))
assert o2.shape == (len(_pp), 4)
S2_PARAMS_M = round(sum(p.numel() for p in _m.parameters()) / 1e6, 1)
print(f"[8] Stage 2 crops {tuple(_x[1].shape)} -> logits {tuple(o2.shape)} OK "
      f"(vit tokens {_m.vit.backbone.patch_embed.num_patches})")
del _m; gc.collect(); torch.cuda.empty_cache()

# 9. resume logic of the REAL run_cv: interrupt -> resume -> skip
_n = len(_pp); _tr = np.arange(_n)[: max(4, _n - 4)]; _va = np.arange(_n)[len(_tr):]
if len(_va) == 0: _va = _tr[-2:]
def _sanity_ds(fold):
    return (CropDataset(_tr, True, pre_arr=_cp, post_arr=_cq, labels=_lab),
            CropDataset(_va, False, pre_arr=_cp, post_arr=_cq, labels=_lab), _va)
_sck = os.path.join(SANITY_DIR, "ckpt")
_mk = lambda: make_ensemble('stage2')
try:
    run_cv('stage2', _n, _lab, _sanity_ds, batch_size=4, accum=1, ckpt_dir=_sck, folds=[0],
           max_epochs=2, make_model=_mk, interrupt_after=1)
    raise AssertionError("interrupt did not fire")
except SimulatedInterruption as e:
    print(f"[9a] {e}")
_p = fold_paths(_sck, 0)
assert os.path.exists(_p['last']) and tload(_p['last'])['resumable'] and not os.path.exists(_p['done'])
r2 = run_cv('stage2', _n, _lab, _sanity_ds, batch_size=4, accum=1, ckpt_dir=_sck, folds=[0],
            max_epochs=2, make_model=_mk)
_d = json.load(open(_p['done']))
assert len(_d['history']['val_f1']) == 2, "resume did not continue at epoch 2"
assert not tload(_p['last'])['resumable'] and os.path.exists(_p['best'])
print("[9b] resumed at epoch 2 and completed the fold; last ckpt slimmed; best ckpt present")
_t0 = time.time()
r3 = run_cv('stage2', _n, _lab, _sanity_ds, batch_size=4, accum=1, ckpt_dir=_sck, folds=[0],
            max_epochs=2, make_model=_mk)
assert r3[0].iloc[0]['best_epoch'] == r2[0].iloc[0]['best_epoch'] and np.array_equal(r3[1][_va], r2[1][_va])
print(f"[9c] completed fold skipped on rerun ({time.time()-_t0:.1f}s, no training)")
sanity['resume'] = {'interrupted_after_epoch': 1, 'resumed_epochs_total': 2, 'skip_on_rerun': True,
                    'last_ckpt_mb': round(os.path.getsize(_p['last']) / 1e6, 1),
                    'best_ckpt_mb': round(os.path.getsize(_p['best']) / 1e6, 1)}

# 10. output directories
assert all(os.path.isdir(p) for p in D.values())
print(f"[10] {len(D)} output directories present under {OUT}")

# 11. ZIP logic (same writer as section 14)
def write_zip(src_dir, zip_path, arc_root, skip_dirs=()):
    n = 0
    with zipfile.ZipFile(zip_path, "w", allowZip64=True) as zf:
        for root, dirs, files in os.walk(src_dir):
            dirs[:] = sorted(d for d in dirs if os.path.join(root, d) not in skip_dirs)
            for fn in sorted(files):
                if fn.endswith(".tmp") or ".tmp." in fn:
                    continue
                full = os.path.join(root, fn)
                comp = zipfile.ZIP_STORED if fn.endswith((".pth", ".npz", ".png", ".zip")) else zipfile.ZIP_DEFLATED
                zf.write(full, os.path.join(arc_root, os.path.relpath(full, src_dir)), compress_type=comp)
                n += 1
    return n

_zp = os.path.join(SANITY_DIR, "sanity_test.zip")
_nz = write_zip(_sck, _zp, "sanity")
with zipfile.ZipFile(_zp) as zf:
    assert zf.testzip() is None
    assert any(x.endswith("fold_1_best.pth") for x in zf.namelist())
print(f"[11] ZIP OK: {_nz} files, CRC verified, checkpoint inside")

sanity['finished'] = datetime.datetime.now().isoformat(timespec='seconds')
sanity['status'] = 'PASSED'
shutil.rmtree(SANITY_DIR, ignore_errors=True)
jdump(sanity, os.path.join(D['repro'], "sanity_check_report.json"))
print("\nSANITY CHECK PASSED")
if SANITY_ONLY:
    raise SystemExit("SANITY_ONLY=True - stopping before training. Set it to False to run the 5-fold CV.")

---
## 7. Stage 1 — hazard classification

Folds are stratified on hazard at the **tile-pair level** (preserved). Grouping by event is impossible here — earthquake and flood each come from a single event — so a very high Stage 1 score partly reflects location; the per-event table quantifies this and it is reported as a benchmark limitation.

In [ ]:
cache_dir = "/kaggle/temp/tile_cache" if os.path.isdir("/kaggle") else os.path.join(OUT, "..", "_tile_cache")
cache_dir = os.path.abspath(cache_dir)
os.makedirs(cache_dir, exist_ok=True)

def cache_one(src, key):
    dst = os.path.join(cache_dir, key + ".jpg")
    if not os.path.exists(dst):
        Image.open(src).convert("RGB").resize((256, 256), Image.BILINEAR).save(dst, quality=95)
    return dst

for df in (dev_df, test_df):
    pre_c, post_c = [], []
    for r in tqdm(df.itertuples(), total=len(df), desc="caching tiles"):
        stem = f"{r.hazard_type}__{r.event_prefix}"
        pre_c.append(cache_one(r.pre_path,  stem + "__pre"))
        post_c.append(cache_one(r.post_path, stem + "__post"))
    df["pre_img"], df["post_img"] = pre_c, post_c
print("cached", 2 * (len(dev_df) + len(test_df)), "tiles in", cache_dir)

### Stage 1 folds — generated once, saved, and reused on every rerun

In [ ]:
S1_FOLD_FILE = os.path.join(D['folds'], "stage1_fold_assignments.csv")
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
_gen = np.full(len(dev_df), -1)
for f, (_, va) in enumerate(skf.split(dev_df, dev_df['label'])):
    _gen[va] = f
if os.path.exists(S1_FOLD_FILE) and RESUME:
    _saved = pd.read_csv(S1_FOLD_FILE)
    _saved = _saved[_saved['split'] == 'dev'].set_index('event_prefix')['fold']
    assert set(_saved.index) == set(dev_df['event_prefix']), "saved Stage 1 folds do not match the data"
    dev_df['fold'] = dev_df['event_prefix'].map(_saved).astype(int)
    print("loaded saved Stage 1 fold assignments:", S1_FOLD_FILE)
    assert (dev_df['fold'].values == _gen).all(), "saved folds differ from the seed-42 regeneration"
else:
    dev_df['fold'] = _gen
assert (dev_df['fold'] >= 0).all()

S1_CHECKSUM = checksum(dev_df['event_prefix'], dev_df['fold'])
FOLD_CHECKSUMS['stage1'] = S1_CHECKSUM
S1_MATCHES_BASELINE = S1_CHECKSUM == EXPECTED_S1_CHECKSUM
print("=" * 64)
print("  STAGE 1 FOLD CHECKSUM :", S1_CHECKSUM,
      "(matches reference notebooks)" if S1_MATCHES_BASELINE else
      f"!! DIFFERS from reference {EXPECTED_S1_CHECKSUM} - baseline comparison is NOT fold-matched")
print("=" * 64)
print(pd.crosstab(dev_df['fold'] + 1, dev_df['hazard_type'], margins=True))

fold_assign = pd.concat([
    dev_df[['event_prefix', 'hazard_type', 'event_name', 'fold']].assign(split='dev'),
    test_df[['event_prefix', 'hazard_type', 'event_name']].assign(fold=-1, split='test'),
])
if not os.path.exists(S1_FOLD_FILE):
    fold_assign.to_csv(S1_FOLD_FILE, index=False)
jdump({'stage1_checksum': S1_CHECKSUM, 'stage1_matches_reference': S1_MATCHES_BASELINE,
       'seed': SEED, 'n_splits': N_SPLITS, 'n_dev_pairs': len(dev_df), 'n_test_pairs': len(test_df)},
      os.path.join(D['folds'], "fold_checksums.json"))

In [ ]:
def s1_make_datasets(fold):
    tr = dev_df[dev_df['fold'] != fold]
    va = dev_df[dev_df['fold'] == fold]
    return (TilePairDataset(tr, s1_train_tf),      # augmented
            TilePairDataset(va, s1_eval_tf),       # deterministic
            dev_df.index[dev_df['fold'] == fold].to_numpy())

if 'stage1' not in FEATURE_DIMS:
    _m = make_ensemble('stage1', pretrained=False); FEATURE_DIMS['stage1'] = feature_dims(_m)
    S1_PARAMS_M = round(sum(p.numel() for p in _m.parameters()) / 1e6, 1); del _m
print("Stage 1 ensemble feature dims:", FEATURE_DIMS['stage1'], "| params (M):", S1_PARAMS_M)

In [ ]:
s1_res, s1_oof, s1_prob, s1_hist = run_cv(
    'stage1', n_items=len(dev_df), labels=dev_df['label'].values,
    make_datasets=s1_make_datasets, batch_size=S1_BATCH, accum=S1_ACCUM)
print()
print(s1_res.to_string(index=False))
s1_res.to_csv(os.path.join(D['metrics'], "stage1_cv_training_summary.csv"), index=False)

### Stage 1 — per-fold results (ensemble)

In [ ]:
y1 = dev_df['label'].values
s1_fold_metrics = []

for f in range(N_SPLITS):
    d = os.path.join(D['s1'], f"fold_{f+1}")
    idx = dev_df.index[dev_df['fold'] == f].to_numpy()
    yt, yp, pr = y1[idx], s1_oof[idx], s1_prob[idx]
    hh = s1_hist[f]
    print("\n" + "#" * 64); print(f"# STAGE 1 (ENSEMBLE) - FOLD {f+1}"); print("#" * 64)
    print(f"best epoch {hh['best_epoch']} | epochs run {hh['epochs_run']}/{MAX_EPOCHS} "
          f"| early stopping {'YES' if hh['early_stopped'] else 'NO'}")
    m = full_report(yt, yp, pr, HAZARD_NAMES, f"Stage 1 ensemble fold {f+1}", d, hist=hh,
                    extra={'fold': f + 1, 'best_epoch': hh['best_epoch'],
                           'epochs_run': hh['epochs_run'], 'early_stopped': hh['early_stopped']})
    s1_fold_metrics.append(m)
    pd.DataFrame({
        'image_id': dev_df.loc[idx, 'event_prefix'].values,
        'event_name': dev_df.loc[idx, 'event_name'].values,
        'hazard': dev_df.loc[idx, 'hazard_type'].values,
        'ground_truth': [HAZARD_NAMES[i] for i in yt],
        'prediction': [HAZARD_NAMES[i] for i in yp],
        'confidence': pr.max(1), 'fold': f + 1,
        **{f'prob_{n}': pr[:, i] for i, n in enumerate(HAZARD_NAMES)},
    }).to_csv(os.path.join(d, "predictions.csv"), index=False)

In [ ]:
S1_KEYS = ['accuracy', 'precision_macro', 'precision_weighted', 'recall_macro', 'recall_weighted',
           'f1_macro', 'f1_weighted', 'roc_auc_macro', 'roc_auc_weighted', 'mcc', 'cohen_kappa']
s1_summary = summary_table(s1_fold_metrics, S1_KEYS)
print("STAGE 1 (ENSEMBLE) - 5-FOLD SUMMARY")
print(s1_summary.round(4).to_string(index=False))
pd.DataFrame(s1_fold_metrics).to_csv(os.path.join(D['metrics'], "stage1_fold_metrics.csv"), index=False)
s1_summary.to_csv(os.path.join(D['metrics'], "stage1_summary.csv"), index=False)
jdump({'folds': s1_fold_metrics, 'summary': s1_summary.to_dict('records')},
      os.path.join(D['metrics'], "stage1_cv_metrics.json"))

# pooled out-of-fold view
assert (s1_oof >= 0).all()
oof1 = pd.DataFrame({'image_id': dev_df['event_prefix'], 'event_name': dev_df['event_name'],
                     'hazard': dev_df['hazard_type'], 'fold': dev_df['fold'] + 1,
                     'ground_truth': [HAZARD_NAMES[i] for i in y1],
                     'prediction': [HAZARD_NAMES[i] for i in s1_oof], 'confidence': s1_prob.max(1),
                     **{f'prob_{n}': s1_prob[:, i] for i, n in enumerate(HAZARD_NAMES)}})
oof1.to_csv(os.path.join(D['oof'], "stage1_oof_predictions.csv"), index=False)
print("\nPooled out-of-fold (all 5 folds combined)")
d_oof = os.path.join(D['s1'], "pooled_oof"); os.makedirs(d_oof, exist_ok=True)
s1_pooled = full_report(y1, s1_oof, s1_prob, HAZARD_NAMES, "Stage 1 ensemble pooled OOF", d_oof)

d1 = dev_df.copy(); d1['correct'] = (s1_oof == y1).astype(int)
ev = (d1.groupby(['hazard_type', 'event_name'])
        .agg(n=('correct', 'size'), accuracy=('correct', 'mean')).round(4))
print("\nAccuracy by disaster event (single-event classes measure location - benchmark limitation):")
print(ev.to_string())
ev.to_csv(os.path.join(D['s1'], "accuracy_by_event.csv"))

---
## 8. Stage 2 — damage severity (group-aware 5-fold CV)

Each building polygon's bounding box, padded 25%, is cut from both the pre and post tile (64 px) and classified into four ordinal severities. `un-classified` is dropped; the majority class is capped at `MAX_PER_CLASS` on the development set with the deterministic MD5 rule (identical crops to the reference runs). Folds: `StratifiedGroupKFold` stratified on severity and **grouped on the source tile** — asserted below.

In [ ]:
dev_poly  = build_polys(dev_df)
test_poly = build_polys(test_df)
print(f"\ndevelopment polygons {len(dev_poly)} | test polygons {len(test_poly)}")
print(dev_poly['subtype'].value_counts().to_string())
print(f"lng_lat present for {test_poly['lat_centroid'].notna().mean():.1%} of test polygons "
      "(taken from the label files; never estimated)")

In [ ]:
def prepare_crops(poly, cap=None):
    s = poly.copy()
    if DROP_UNCLASSIFIED:
        s = s[s['subtype'].isin(DAMAGE_NAMES)]
    s['label'] = s['subtype'].map(damage_to_idx)
    if cap:
        s['_r'] = s['crop_id'].map(lambda t: hashlib.md5(t.encode()).hexdigest())
        s = (s.sort_values(['label', '_r']).groupby('label', group_keys=False)
               .head(cap).sort_values('_r').drop(columns=['_r']))
    return s.reset_index(drop=True)

sel      = prepare_crops(dev_poly,  cap=MAX_PER_CLASS)   # CV pool
test_sel = prepare_crops(test_poly, cap=None)            # natural distribution
sel['slot'] = sel.index
test_sel['slot'] = test_sel.index

print(f"development crops {len(sel)} (capped at {MAX_PER_CLASS}/class)")
print(sel['subtype'].value_counts().to_string())
print(f"\ntest crops {len(test_sel)} (uncapped, natural distribution)")
print(test_sel['subtype'].value_counts().to_string())

In [ ]:
S2_FOLD_FILE = os.path.join(D['folds'], "stage2_fold_assignments.csv")
sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
_gen2 = np.full(len(sel), -1)
for f, (_, va) in enumerate(sgkf.split(sel, y=sel['label'], groups=sel['event_prefix'])):
    _gen2[va] = f
if os.path.exists(S2_FOLD_FILE) and RESUME:
    _saved2 = pd.read_csv(S2_FOLD_FILE).set_index('crop_id')['fold']
    assert set(_saved2.index) == set(sel['crop_id']), "saved Stage 2 folds do not match the crops"
    sel['fold'] = sel['crop_id'].map(_saved2).astype(int)
    print("loaded saved Stage 2 fold assignments:", S2_FOLD_FILE)
    assert (sel['fold'].values == _gen2).all(), "saved folds differ from the seed-42 regeneration"
else:
    sel['fold'] = _gen2
assert (sel['fold'] >= 0).all()

for f in range(N_SPLITS):
    tr_g = set(sel.loc[sel.fold != f, 'event_prefix'])
    va_g = set(sel.loc[sel.fold == f, 'event_prefix'])
    assert not (tr_g & va_g), f"group leakage in fold {f+1}"
print("[OK] No group leakage - no tile appears in both train and validation of any fold")
assert not (set(sel['event_prefix']) & set(test_sel['event_prefix'])), "test tiles leaked into the CV pool"
print("[OK] Test set untouched - test crops come only from the held-out tiles")

S2_CHECKSUM = checksum(sel['crop_id'], sel['fold'])
FOLD_CHECKSUMS['stage2'] = S2_CHECKSUM
S2_MATCHES_BASELINE = S2_CHECKSUM == EXPECTED_S2_CHECKSUM
print("\n" + "=" * 64)
print("  STAGE 2 FOLD CHECKSUM :", S2_CHECKSUM,
      "(matches reference notebooks)" if S2_MATCHES_BASELINE else
      f"!! DIFFERS from reference {EXPECTED_S2_CHECKSUM} - baseline comparison is NOT fold-matched")
print("=" * 64)
print(pd.crosstab(sel['fold'] + 1, sel['subtype'], margins=True))

if not os.path.exists(S2_FOLD_FILE):
    sel[['crop_id', 'event_prefix', 'hazard_type', 'event_name', 'subtype', 'label', 'fold']].to_csv(
        S2_FOLD_FILE, index=False)
test_sel[['crop_id', 'event_prefix', 'hazard_type', 'event_name', 'subtype', 'label']].assign(fold=-1).to_csv(
    os.path.join(D['folds'], "stage2_test_crops.csv"), index=False)
cs = json.load(open(os.path.join(D['folds'], "fold_checksums.json")))
cs.update({'stage2_checksum': S2_CHECKSUM, 'stage2_matches_reference': S2_MATCHES_BASELINE,
           'n_dev_crops': len(sel), 'n_test_crops': len(test_sel)})
jdump(cs, os.path.join(D['folds'], "fold_checksums.json"))

In [ ]:
TILE_SIZES = {}

def extract(s):
    ap = np.zeros((len(s), S2_CROP, S2_CROP, 3), np.uint8)
    aq = np.zeros((len(s), S2_CROP, S2_CROP, 3), np.uint8)
    boxes = np.zeros((len(s), 4), np.int32)
    for prefix, grp in tqdm(s.groupby('event_prefix'), total=s['event_prefix'].nunique(),
                            desc="cropping buildings"):
        r0 = grp.iloc[0]
        po = Image.open(r0['post_path']).convert('RGB')
        pr = Image.open(r0['pre_path']).convert('RGB')
        W, H = po.size
        TILE_SIZES[prefix] = (W, H)
        for r in grp.itertuples():
            b = padded_box(r.x0, r.y0, r.x1, r.y1, W, H)
            boxes[r.slot] = b
            aq[r.slot] = np.asarray(po.crop(b).resize((S2_CROP, S2_CROP), Image.BILINEAR))
            ap[r.slot] = np.asarray(pr.crop(b).resize((S2_CROP, S2_CROP), Image.BILINEAR))
    return ap, aq, boxes

arr_pre, arr_post, _ = extract(sel)
t_pre, t_post, t_boxes = extract(test_sel)
test_sel[['crop_x0', 'crop_y0', 'crop_x1', 'crop_y1']] = t_boxes
print("dev crops:", arr_post.shape, "| test crops:", t_post.shape)
print(f"memory: {(arr_post.nbytes*2 + t_post.nbytes*2)/1e9:.2f} GB")

def s2_make_datasets(fold):
    tr = sel.index[sel['fold'] != fold].to_numpy()
    va = sel.index[sel['fold'] == fold].to_numpy()
    return CropDataset(tr, True), CropDataset(va, False), va

if 'stage2' not in FEATURE_DIMS:
    _m = make_ensemble('stage2', pretrained=False); FEATURE_DIMS['stage2'] = feature_dims(_m)
    S2_PARAMS_M = round(sum(p.numel() for p in _m.parameters()) / 1e6, 1); del _m
print("Stage 2 ensemble feature dims:", FEATURE_DIMS['stage2'], "| params (M):", S2_PARAMS_M)

In [ ]:
s2_res, s2_oof, s2_prob, s2_hist = run_cv(
    'stage2', n_items=len(sel), labels=sel['label'].values,
    make_datasets=s2_make_datasets, batch_size=S2_BATCH, accum=S2_ACCUM)
print()
print(s2_res.to_string(index=False))
s2_res.to_csv(os.path.join(D['metrics'], "stage2_cv_training_summary.csv"), index=False)

### 8A. Multi-class severity — per-fold results (QWK included: the classes are ordinal)

In [ ]:
y2 = sel['label'].values
s2_fold_metrics = []

for f in range(N_SPLITS):
    d = os.path.join(D['s2'], f"fold_{f+1}")
    idx = sel.index[sel['fold'] == f].to_numpy()
    yt, yp, pr = y2[idx], s2_oof[idx], s2_prob[idx]
    hh = s2_hist[f]
    print("\n" + "#" * 64); print(f"# STAGE 2 MULTI-CLASS (ENSEMBLE) - FOLD {f+1}"); print("#" * 64)
    print(f"best epoch {hh['best_epoch']} | epochs run {hh['epochs_run']}/{MAX_EPOCHS} "
          f"| early stopping {'YES' if hh['early_stopped'] else 'NO'}")
    err = yt != yp
    m = full_report(yt, yp, pr, DAMAGE_NAMES, f"Stage 2 ensemble fold {f+1}", d, ordinal=True, hist=hh,
                    extra={'fold': f + 1, 'best_epoch': hh['best_epoch'], 'epochs_run': hh['epochs_run'],
                           'early_stopped': hh['early_stopped'],
                           'adjacent_error_rate': float((np.abs(yt - yp)[err] == 1).mean()) if err.any() else 0.0})
    s2_fold_metrics.append(m)
    p_dmg = pr[:, 1] + pr[:, 2] + pr[:, 3]
    pd.DataFrame({
        'crop_id': sel.loc[idx, 'crop_id'].values,
        'parent_image_id': sel.loc[idx, 'event_prefix'].values,
        'hazard': sel.loc[idx, 'hazard_type'].values,
        'ground_truth': [DAMAGE_NAMES[i] for i in yt],
        'prediction': [DAMAGE_NAMES[i] for i in yp],
        'confidence': pr.max(1), 'prob_damaged': p_dmg, 'fold': f + 1,
        **{f'prob_{n}': pr[:, i] for i, n in enumerate(DAMAGE_NAMES)},
    }).to_csv(os.path.join(d, "predictions.csv"), index=False)

### 8B. Binary severity — derived, not retrained

`damaged = minor ∪ major ∪ destroyed`, `P(damaged) = P(minor) + P(major) + P(destroyed)`, predicted damaged when `P(damaged) > 0.5` (fixed, never tuned).

In [ ]:
s2b_fold_metrics = []
for f in range(N_SPLITS):
    d = os.path.join(D['s2b'], f"fold_{f+1}")
    idx = sel.index[sel['fold'] == f].to_numpy()
    yb, pb_pred, pb = to_binary(y2[idx], s2_prob[idx])
    print("\n" + "#" * 64); print(f"# STAGE 2 BINARY (ENSEMBLE) - FOLD {f+1}"); print("#" * 64)
    m = full_report(yb, pb_pred, pb, BINARY_NAMES, f"Binary severity ensemble fold {f+1}", d,
                    extra={'fold': f + 1})
    s2b_fold_metrics.append(m)
    pd.DataFrame({
        'crop_id': sel.loc[idx, 'crop_id'].values,
        'ground_truth': [BINARY_NAMES[i] for i in yb],
        'prediction': [BINARY_NAMES[i] for i in pb_pred],
        'prob_damaged': pb[:, 1], 'confidence': pb.max(1), 'fold': f + 1,
    }).to_csv(os.path.join(d, "predictions.csv"), index=False)

In [ ]:
S2_KEYS  = ['accuracy', 'precision_macro', 'precision_weighted', 'recall_macro', 'recall_weighted',
            'f1_macro', 'f1_weighted', 'roc_auc_macro', 'roc_auc_weighted', 'qwk', 'mcc', 'cohen_kappa',
            'adjacent_error_rate']
S2B_KEYS = ['accuracy', 'precision_macro', 'precision_weighted', 'recall_macro', 'recall_weighted',
            'f1_macro', 'f1_weighted', 'roc_auc', 'mcc', 'cohen_kappa']
s2_summary  = summary_table(s2_fold_metrics,  S2_KEYS)
s2b_summary = summary_table(s2b_fold_metrics, S2B_KEYS)
print("STAGE 2 MULTI-CLASS (ENSEMBLE) - 5-FOLD SUMMARY")
print(s2_summary.round(4).to_string(index=False))
print("\nSTAGE 2 BINARY (ENSEMBLE) - 5-FOLD SUMMARY")
print(s2b_summary.round(4).to_string(index=False))

pd.DataFrame(s2_fold_metrics).to_csv(os.path.join(D['metrics'], "stage2_multiclass_fold_metrics.csv"), index=False)
s2_summary.to_csv(os.path.join(D['metrics'], "stage2_multiclass_summary.csv"), index=False)
pd.DataFrame(s2b_fold_metrics).to_csv(os.path.join(D['metrics'], "stage2_binary_fold_metrics.csv"), index=False)
s2b_summary.to_csv(os.path.join(D['metrics'], "stage2_binary_summary.csv"), index=False)
jdump({'multiclass': {'folds': s2_fold_metrics, 'summary': s2_summary.to_dict('records')},
       'binary': {'folds': s2b_fold_metrics, 'summary': s2b_summary.to_dict('records')}},
      os.path.join(D['metrics'], "stage2_cv_metrics.json"))

assert (s2_oof >= 0).all()
yb_all, ybp_all, pb_all = to_binary(y2, s2_prob)
oof2 = pd.DataFrame({'crop_id': sel['crop_id'], 'parent_image_id': sel['event_prefix'],
                     'hazard': sel['hazard_type'], 'fold': sel['fold'] + 1,
                     'ground_truth': [DAMAGE_NAMES[i] for i in y2],
                     'prediction': [DAMAGE_NAMES[i] for i in s2_oof], 'confidence': s2_prob.max(1),
                     **{f'prob_{n}': s2_prob[:, i] for i, n in enumerate(DAMAGE_NAMES)},
                     'prob_damaged': pb_all[:, 1],
                     'gt_binary': [BINARY_NAMES[i] for i in yb_all],
                     'pred_binary': [BINARY_NAMES[i] for i in ybp_all]})
oof2.to_csv(os.path.join(D['oof'], "stage2_oof_predictions.csv"), index=False)
d_oof = os.path.join(D['s2'], "pooled_oof"); os.makedirs(d_oof, exist_ok=True)
s2_pooled = full_report(y2, s2_oof, s2_prob, DAMAGE_NAMES, "Stage 2 ensemble pooled OOF", d_oof, ordinal=True)
d_oofb = os.path.join(D['s2b'], "pooled_oof"); os.makedirs(d_oofb, exist_ok=True)
s2b_pooled = full_report(yb_all, ybp_all, pb_all, BINARY_NAMES, "Binary severity ensemble pooled OOF", d_oofb)

q = [m['qwk'] for m in s2_fold_metrics]
print(f"\nQWK per fold: {np.round(q, 4).tolist()} | mean {np.mean(q):.4f} +/- {np.std(q, ddof=1) if len(q)>1 else 0:.4f}")

---
## 9. Explainability — Grad-CAM, one explanation per branch (preserved methodology)

The reference Grad-CAM is reused unchanged: hooks on a target layer, gradient of the predicted class score, channel-weighted sum, ReLU, upsample, min-max normalise; transformer tokens are reshaped to their patch grid (class token dropped). For the ensemble the predicted class score is the **fused** logit, and Grad-CAM is computed separately for each branch's own target layer:

| branch | Stage 1 target (224 px) | Stage 2 target (64 px crops) |
|---|---|---|
| ConvNeXt | `conv.backbone.stages[3]` → 7×7 | `conv.backbone.stages[1]` → 8×8 |
| ViT | `vit.backbone.blocks[-1].norm1` → 14×14 tokens | same → 4×4 tokens |

The two maps answer *"which pixels did this branch use for the ensemble's decision"*. They are saved as **separate** ConvNeXt and ViT explanations; no combined/averaged ensemble heatmap is produced, because such a map would not correspond to any quantity the model computes. Each branch calls the post image first, so the first captured activation and the last captured gradient belong to the post-disaster image. Samples: fold-1 model on fold-1 validation data (as in the reference notebooks).

In [ ]:
def vit_reshape(n_prefix=1):
    """(B, N, C) token sequence -> (B, C, h, w) grid, dropping the prefix (class) tokens."""
    def f(t):
        t = t[:, n_prefix:, :]
        b, n, c = t.shape
        h = int(round(n ** 0.5))
        assert h * h == n, f"{n} patch tokens is not a square grid"
        return t.reshape(b, h, h, c).permute(0, 3, 1, 2).contiguous()
    return f


class GradCAM:
    """Grad-CAM for one branch of the siamese fusion model, attributed to the POST image."""
    def __init__(self, model, target_layer, reshape=None):
        self.model = model
        self.reshape = reshape
        self.acts, self.grads = [], []
        self.h1 = target_layer.register_forward_hook(
            lambda m, i, o: self.acts.append(o.detach()))
        self.h2 = target_layer.register_full_backward_hook(
            lambda m, gi, go: self.grads.append(go[0].detach()))

    def remove(self):
        self.h1.remove(); self.h2.remove()

    def __call__(self, pre, post, cls=None):
        self.acts.clear(); self.grads.clear()
        self.model.zero_grad()
        out = self.model(pre, post)                     # post called first in each branch
        prob = F.softmax(out.float(), 1)
        if cls is None:
            cls = out.argmax(1)
        out.gather(1, cls.view(-1, 1)).sum().backward()
        A = self.acts[0]                                # forward order: post, then pre
        G = self.grads[-1]                              # backward is reversed -> post last
        if self.reshape is not None:
            A, G = self.reshape(A), self.reshape(G)
        w = G.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((w * A).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=post.shape[-2:], mode='bilinear', align_corners=False)
        cam = cam.squeeze(1)
        flat = cam.flatten(1)
        cam = ((cam - flat.min(1)[0].view(-1, 1, 1)) /
               (flat.max(1)[0].view(-1, 1, 1) - flat.min(1)[0].view(-1, 1, 1) + 1e-8))
        return cam.cpu().numpy(), cls.cpu().numpy(), prob.detach().cpu().numpy()


def denorm(t):
    x = t.cpu() * _STD_T + _MEAN_T
    return x.permute(1, 2, 0).numpy().clip(0, 1)


def branch_targets(model, stage):
    conv_layer = model.conv.backbone.stages[3 if stage == 'stage1' else 1]
    npfx = getattr(model.vit.backbone, 'num_prefix_tokens', 1)
    return {'convnext': (conv_layer, None), 'vit': (model.vit.backbone.blocks[-1].norm1, vit_reshape(npfx))}


def branch_gradcams(model, stage, samples, names, title, out_dir, tag):
    """samples: list of (pre_tensor, post_tensor, true_label, tag, sample_id).
    Saves one panel + raw maps per branch; returns {branch: cams}."""
    results = {}
    for branch, (layer, reshape) in branch_targets(model, stage).items():
        eng = GradCAM(model, layer, reshape=reshape)
        rows = len(samples)
        fig, axes = plt.subplots(rows, 4, figsize=(13, 3.1 * rows))
        if rows == 1:
            axes = axes[None, :]
        cams, records = [], []
        for r, (pre, post, yt, stag, sid) in enumerate(samples):
            cam, cls, prob = eng(pre.unsqueeze(0).to(device), post.unsqueeze(0).to(device))
            cam, cls, conf = cam[0], int(cls[0]), float(prob[0].max())
            cams.append(cam)
            records.append({'sample_id': sid, 'selection': stag, 'true': names[yt], 'pred': names[cls],
                            'confidence': conf, 'correct': cls == yt,
                            'probabilities': {n: float(prob[0][i]) for i, n in enumerate(names)}})
            pre_i, post_i = denorm(pre), denorm(post)
            axes[r, 0].imshow(pre_i);  axes[r, 0].set_title("pre-disaster", fontsize=9)
            axes[r, 1].imshow(post_i); axes[r, 1].set_title("post-disaster", fontsize=9)
            axes[r, 2].imshow(cam, cmap='jet'); axes[r, 2].set_title(f"{branch} Grad-CAM", fontsize=9)
            axes[r, 3].imshow(post_i); axes[r, 3].imshow(cam, cmap='jet', alpha=0.45)
            axes[r, 3].set_title(f"{branch} overlay", fontsize=9)
            axes[r, 0].set_ylabel(f"{stag}\ntrue: {names[yt]}\npred: {names[cls]} ({conf:.2f})\n"
                                  f"{'CORRECT' if cls == yt else 'WRONG'}", fontsize=8)
            for c in range(4):
                axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
        eng.remove()
        plt.suptitle(f"{title} — {branch.upper()} branch explanation of the fused prediction", y=1.005)
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, branch, f"{tag}_{branch}_gradcam.png"), dpi=150, bbox_inches='tight')
        plt.show(); plt.close(fig)
        np.savez_compressed(os.path.join(out_dir, branch, f"{tag}_{branch}_gradcam_maps.npz"),
                            cams=np.stack(cams), sample_ids=np.array([s[4] for s in samples]))
        jdump({'branch': branch, 'stage': stage, 'target_layer': str(layer.__class__.__name__),
               'map_resolution_note': 'stage1: convnext 7x7 / vit 14x14; stage2: convnext 8x8 / vit 4x4',
               'samples': records}, os.path.join(out_dir, branch, f"{tag}_{branch}_gradcam.json"))
        results[branch] = np.stack(cams)
    return results

print("Grad-CAM ready")

In [ ]:
XAI_S2 = os.path.join(D['xai'], 'stage2')
model = make_ensemble('stage2', pretrained=False).to(device)
load_weights(model, fold_paths(D['ckpt_s2'], 0)['best'])
model.eval()

idx  = sel.index[sel['fold'] == 0].to_numpy()
yt_, yp_, pr_ = y2[idx], s2_oof[idx], s2_prob[idx]
conf_ = pr_.max(1)
correct = yt_ == yp_
picks = []
if correct.any():
    j = idx[np.where(correct)[0][int(np.argmax(conf_[correct]))]]
    picks.append((j, "high-confidence correct"))
    cj = np.where(correct)[0]
    picks.append((idx[cj[len(cj) // 2]], "typical correct"))
if (~correct).any():
    w = np.where(~correct)[0]
    picks.append((idx[w[int(np.argmax(conf_[~correct]))]], "high-confidence WRONG"))
    picks.append((idx[w[len(w) // 2]], "typical incorrect"))

ds = CropDataset([p[0] for p in picks], train=False)
samples_s2 = [(ds[i][0], ds[i][1], int(ds[i][2]), picks[i][1], sel.loc[picks[i][0], 'crop_id'])
              for i in range(len(picks))]
xai_s2 = branch_gradcams(model, 'stage2', samples_s2, DAMAGE_NAMES, "Stage 2 severity (fold 1)",
                         XAI_S2, "stage2_severity")
del model; gc.collect(); torch.cuda.empty_cache()

In [ ]:
XAI_S1 = os.path.join(D['xai'], 'stage1')
model = make_ensemble('stage1', pretrained=False).to(device)
load_weights(model, fold_paths(D['ckpt_s1'], 0)['best'])
model.eval()

idx = dev_df.index[dev_df['fold'] == 0].to_numpy()
yt_ = y1[idx]
picks = []
for h in range(4):
    cand = idx[(yt_ == h)]
    if len(cand):
        picks.append((cand[0], f"{HAZARD_NAMES[h]} sample"))
ds = TilePairDataset(dev_df.loc[[p[0] for p in picks]], s1_eval_tf)
samples_s1 = [(ds[i][0], ds[i][1], int(ds[i][2]), picks[i][1], dev_df.loc[picks[i][0], 'event_prefix'])
              for i in range(len(picks))]
xai_s1 = branch_gradcams(model, 'stage1', samples_s1, HAZARD_NAMES, "Stage 1 hazard (fold 1)",
                         XAI_S1, "stage1_hazard")
del model; gc.collect(); torch.cuda.empty_cache()

# side-by-side view: the two SEPARATE branch maps next to each other (not fused)
def side_by_side(res, samples_, names_, path, title):
    rows = len(samples_)
    fig, axes = plt.subplots(rows, 3, figsize=(10, 3.1 * rows))
    if rows == 1: axes = axes[None, :]
    for r, s in enumerate(samples_):
        post_i = denorm(s[1])
        axes[r, 0].imshow(post_i); axes[r, 0].set_title("post-disaster", fontsize=9)
        for c, br in enumerate(['convnext', 'vit'], start=1):
            axes[r, c].imshow(post_i); axes[r, c].imshow(res[br][r], cmap='jet', alpha=0.45)
            axes[r, c].set_title(f"{br} branch (separate map)", fontsize=9)
        axes[r, 0].set_ylabel(f"{s[3]}\ntrue: {names_[s[2]]}", fontsize=8)
        for a in axes[r]: a.set_xticks([]); a.set_yticks([])
    plt.suptitle(title, y=1.005); plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight'); plt.show(); plt.close(fig)

side_by_side(xai_s1, samples_s1, HAZARD_NAMES, os.path.join(XAI_S1, "stage1_branch_maps_side_by_side.png"),
             "Stage 1 — ConvNeXt vs ViT branch Grad-CAM (two separate explanations)")
side_by_side(xai_s2, samples_s2, DAMAGE_NAMES, os.path.join(XAI_S2, "stage2_branch_maps_side_by_side.png"),
             "Stage 2 — ConvNeXt vs ViT branch Grad-CAM (two separate explanations)")

---
## 10. Final ensemble — trained on the complete development set

A fresh ensemble (ImageNet weights) is trained per stage on **all** development data for a fixed number of epochs equal to the **mean CV best epoch** (rounded) — selected from development data only, the reference refit protocol. Resumable per epoch; the result is `checkpoints/final/stage{1,2}_ensemble_final.pth`.

In [ ]:
S1_FINAL_EPOCHS = max(1, int(round(np.mean([h['best_epoch'] for h in s1_hist.values()]))))
S2_FINAL_EPOCHS = max(1, int(round(np.mean([h['best_epoch'] for h in s2_hist.values()]))))
print(f"Stage 1 best epochs per fold: {[h['best_epoch'] for h in s1_hist.values()]} -> final {S1_FINAL_EPOCHS} epochs")
print(f"Stage 2 best epochs per fold: {[h['best_epoch'] for h in s2_hist.values()]} -> final {S2_FINAL_EPOCHS} epochs")


def train_final(stage, train_ds, labels, epochs, batch_size, accum, cv_hist, cv_summary):
    final_path  = os.path.join(D['ckpt_final'], f"{stage}_ensemble_final.pth")
    resume_path = os.path.join(D['ckpt_final'], f"{stage}_final_resume.pth")
    if FORCE_RETRAIN:
        for p in (final_path, resume_path):
            if os.path.exists(p): os.remove(p)
    if RESUME and os.path.exists(final_path):
        meta = tload(final_path)['meta']
        print(f"  [RESUME] final {stage} model already trained ({meta['epoch']} epochs) - not retrained")
        return final_path, meta
    n_classes = len(STAGE_SPEC[stage]['class_names'])
    set_seed(SEED)
    model = make_ensemble(stage).to(device)
    ld = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                    num_workers=NUM_WORKERS, pin_memory=True, drop_last=len(train_ds) > batch_size)
    w = compute_class_weight('balanced', classes=np.arange(n_classes), y=labels)
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32, device=device),
                                    label_smoothing=LABEL_SMOOTH)
    opt = torch.optim.AdamW(param_groups(model), weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    hist, start = {'train_loss': [], 'train_acc': [], 'train_f1': []}, 1
    if RESUME and os.path.exists(resume_path):
        ck = tload(resume_path)
        model.load_state_dict(ck['state_dict']); opt.load_state_dict(ck['optimizer'])
        sched.load_state_dict(ck['scheduler']); scaler.load_state_dict(ck['scaler'])
        hist, start = ck['history'], ck['epoch'] + 1; restore_rng(ck['rng'])
        print(f"  [RESUME] final {stage} model continues at epoch {start}/{epochs}"); del ck
    for ep in range(start, epochs + 1):
        trl, tra, trf, _, _, _ = run_epoch(model, ld, criterion, opt, scaler,
                                           desc=f"{stage} final e{ep}", accum=accum)
        sched.step()
        hist['train_loss'].append(trl); hist['train_acc'].append(tra); hist['train_f1'].append(trf)
        atomic_torch_save({'epoch': ep, 'state_dict': model.state_dict(), 'optimizer': opt.state_dict(),
                           'scheduler': sched.state_dict(), 'scaler': scaler.state_dict(),
                           'history': hist, 'rng': capture_rng()}, resume_path)
        print(f"  epoch {ep:2d}/{epochs} | train loss {trl:.4f} acc {tra:.4f} macroF1 {trf:.4f}")
    meta = ckpt_meta(stage, 'final', epochs, None, 'final', extra={
        'trained_on': 'complete development set (all CV folds)', 'n_train': len(train_ds),
        'final_epochs': epochs, 'epoch_rule': 'round(mean CV best epoch)',
        'cv_best_epochs': [h['best_epoch'] for h in cv_hist.values()],
        'cv_val_macro_f1_mean': float(cv_summary.set_index('Metric').loc['f1_macro', 'Mean']),
        'history': hist})
    atomic_torch_save({'state_dict': cpu_state(model), 'meta': meta}, final_path)
    if os.path.exists(resume_path): os.remove(resume_path)
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    ax[0].plot(range(1, len(hist['train_loss']) + 1), hist['train_loss'], marker='o'); ax[0].set_title(f'{stage} final - train loss')
    ax[1].plot(range(1, len(hist['train_f1']) + 1), hist['train_f1'], marker='o'); ax[1].set_title(f'{stage} final - train macro-F1')
    for a in ax: a.set_xlabel('epoch'); a.grid(alpha=.3)
    plt.tight_layout(); plt.savefig(os.path.join(D['final_test'], f"{stage}_final_training_curve.png"), dpi=150)
    plt.show(); plt.close(fig)
    pd.DataFrame(hist).to_csv(os.path.join(D['final_test'], f"{stage}_final_training_history.csv"), index=False)
    del model, opt, sched, scaler, ld; gc.collect(); torch.cuda.empty_cache()
    return final_path, meta


@torch.no_grad()
def evaluate(model, ds, batch_size):
    model.eval()
    ld = DataLoader(ds, batch_size=batch_size * 2, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=True)
    P, Y = [], []
    for pre, post, y in tqdm(ld, desc="test inference", leave=False):
        with torch.amp.autocast('cuda', enabled=AMP):
            out = model(pre.to(device), post.to(device))
        P.append(F.softmax(out.float(), 1).cpu().numpy()); Y.append(y.numpy())
    P, Y = np.concatenate(P), np.concatenate(Y)
    return Y, P.argmax(1), P


def test_once(stage, final_path, final_meta, ds, batch_size):
    """The held-out test set is scored exactly once per final model; reruns reuse the stored outputs."""
    cache = os.path.join(D['final_test'], f"{stage}_test_outputs.npz")
    if os.path.exists(cache):
        z = np.load(cache, allow_pickle=True)
        if str(z['model_created']) == final_meta['created']:
            print(f"  [RESUME] {stage} test set already evaluated with this final model - reusing outputs")
            return z['y'], z['pred'], z['prob']
    model = make_ensemble(stage, pretrained=False).to(device)
    load_weights(model, final_path)
    y, pred, prob = evaluate(model, ds, batch_size)
    np.savez(cache, y=y, pred=pred, prob=prob, model_created=final_meta['created'])
    jdump({'stage': stage, 'evaluated_at': datetime.datetime.now().isoformat(timespec='seconds'),
           'final_checkpoint': os.path.relpath(final_path, OUT), 'n_test': int(len(y))},
          os.path.join(D['final_test'], f"{stage}_test_evaluation_log.json"))
    del model; gc.collect(); torch.cuda.empty_cache()
    return y, pred, prob

In [ ]:
print("=" * 64); print("FINAL STAGE 1 ENSEMBLE - training on the full development set"); print("=" * 64)
S1_FINAL_PATH, S1_FINAL_META = train_final('stage1', TilePairDataset(dev_df, s1_train_tf), dev_df['label'].values,
                                           S1_FINAL_EPOCHS, S1_BATCH, S1_ACCUM, s1_hist, s1_summary)
print("=" * 64); print("FINAL STAGE 2 ENSEMBLE - training on the full development set"); print("=" * 64)
S2_FINAL_PATH, S2_FINAL_META = train_final('stage2', CropDataset(sel.index.to_numpy(), True), sel['label'].values,
                                           S2_FINAL_EPOCHS, S2_BATCH, S2_ACCUM, s2_hist, s2_summary)

---
## 11. FINAL HELD-OUT TEST EVALUATION (once)

Everything above is **5-FOLD CROSS-VALIDATION**; everything below is **FINAL HELD-OUT TEST**. Deterministic preprocessing only — no augmentation, no TTA, no threshold tuning. Stage 2 is reported on the natural distribution, on the same per-class cap rule as the CV pool (comparable view), and as the derived binary task.

In [ ]:
FT = {k: os.path.join(D['final_test'], k) for k in
      ['stage1_hazard', 'stage2_multiclass_natural', 'stage2_multiclass_capped', 'stage2_binary']}
for p in FT.values(): os.makedirs(p, exist_ok=True)

yt1, yp1, pr1 = test_once('stage1', S1_FINAL_PATH, S1_FINAL_META, TilePairDataset(test_df, s1_eval_tf), S1_BATCH)
print("\n" + "=" * 64); print("FINAL HELD-OUT TEST RESULTS - STAGE 1 HAZARD (ENSEMBLE)"); print("=" * 64)
t1 = full_report(yt1, yp1, pr1, HAZARD_NAMES, "FINAL TEST - Stage 1 hazard", FT['stage1_hazard'])
s1_test_pred = pd.DataFrame({'image_id': test_df['event_prefix'].values, 'event_name': test_df['event_name'].values,
              'hazard': test_df['hazard_type'].values,
              'ground_truth': [HAZARD_NAMES[i] for i in yt1], 'prediction': [HAZARD_NAMES[i] for i in yp1],
              'confidence': pr1.max(1), **{f'prob_{n}': pr1[:, i] for i, n in enumerate(HAZARD_NAMES)}})
s1_test_pred.to_csv(os.path.join(D['final_test'], "stage1_test_predictions.csv"), index=False)

In [ ]:
test_ds = CropDataset(np.arange(len(test_sel)), False, pre_arr=t_pre, post_arr=t_post,
                      labels=test_sel['label'].values)
yt2, yp2, pr2 = test_once('stage2', S2_FINAL_PATH, S2_FINAL_META, test_ds, S2_BATCH)

print("\n" + "=" * 64); print("FINAL HELD-OUT TEST - STAGE 2 MULTI-CLASS (natural distribution)"); print("=" * 64)
t2 = full_report(yt2, yp2, pr2, DAMAGE_NAMES, "FINAL TEST - Stage 2 severity (natural)",
                 FT['stage2_multiclass_natural'], ordinal=True)

capped = prepare_crops(test_poly, cap=MAX_PER_CLASS)
mask = test_sel['crop_id'].isin(set(capped['crop_id'])).values
print(f"\nSame test set with the CV per-class cap rule applied ({mask.sum()} crops):")
t2c = full_report(yt2[mask], yp2[mask], pr2[mask], DAMAGE_NAMES, "FINAL TEST - Stage 2 severity (capped)",
                  FT['stage2_multiclass_capped'], ordinal=True)

yb2, pbp2, pb2 = to_binary(yt2, pr2)
print("\n" + "=" * 64); print("FINAL HELD-OUT TEST - STAGE 2 BINARY (derived)"); print("=" * 64)
t2b = full_report(yb2, pbp2, pb2, BINARY_NAMES, "FINAL TEST - binary severity", FT['stage2_binary'])

s1_tile_pred = s1_test_pred.set_index('image_id')
s2_test_pred = pd.DataFrame({
    'crop_id': test_sel['crop_id'].values, 'building_uid': test_sel['uid'].values,
    'parent_image_id': test_sel['event_prefix'].values, 'event_name': test_sel['event_name'].values,
    'hazard': test_sel['hazard_type'].values,
    'predicted_hazard': s1_tile_pred.loc[test_sel['event_prefix'], 'prediction'].values,
    'ground_truth': [DAMAGE_NAMES[i] for i in yt2], 'prediction': [DAMAGE_NAMES[i] for i in yp2],
    'gt_binary': [BINARY_NAMES[i] for i in yb2], 'pred_binary': [BINARY_NAMES[i] for i in pbp2],
    'confidence': pr2.max(1), 'prob_damaged': pb2[:, 1],
    **{f'prob_{n}': pr2[:, i] for i, n in enumerate(DAMAGE_NAMES)},
    'in_capped_view': mask})
s2_test_pred.to_csv(os.path.join(D['final_test'], "stage2_test_predictions.csv"), index=False)
np.savez_compressed(os.path.join(D['final_test'], "test_probabilities.npz"),
                    stage1_prob=pr1, stage1_y=yt1, stage2_prob=pr2, stage2_y=yt2,
                    stage1_ids=test_df['event_prefix'].values, stage2_ids=test_sel['crop_id'].values)

final_test_metrics = pd.DataFrame([{'stage': 'stage1_hazard', **t1},
                                   {'stage': 'stage2_multiclass_natural', **t2},
                                   {'stage': 'stage2_multiclass_capped', **t2c},
                                   {'stage': 'stage2_binary', **t2b}])
final_test_metrics.to_csv(os.path.join(D['metrics'], "final_test_metrics.csv"), index=False)
jdump(final_test_metrics.to_dict('records'), os.path.join(D['metrics'], "final_test_metrics.json"))
print(final_test_metrics.round(4).to_string(index=False))

---
## 12. Baseline comparison — existing ConvNeXt, existing ViT, NEW trained ensemble

The baselines are **not retrained**. Their numbers come from the existing notebooks: if their result folders (`convnext_results/`, `vit_results/`) are attached under `/kaggle/input`, the CSVs are read; otherwise the values printed in those notebooks (embedded below, 4 d.p.) are used. All three models share the seed-42 folds — the checksum check states whether that holds for this run.

**Ablation (baseline, NOT the proposed model):** simple probability soft-voting of the existing ConvNeXt and ViT predictions. It needs the existing per-fold and test `predictions.csv` files, so it runs only when those folders are attached.

In [ ]:
EMBEDDED_BASELINES = json.loads(r'''{
 "convnext": {
  "cv": {
   "stage1_hazard": {
    "accuracy": {
     "folds": [
      0.9953,
      1.0,
      1.0,
      0.9977,
      1.0
     ],
     "mean": 0.9986,
     "std": 0.0021
    },
    "precision_macro": {
     "folds": [
      0.9866,
      1.0,
      1.0,
      0.9987,
      1.0
     ],
     "mean": 0.9971,
     "std": 0.0059
    },
    "recall_macro": {
     "folds": [
      0.9974,
      1.0,
      1.0,
      0.9868,
      1.0
     ],
     "mean": 0.9969,
     "std": 0.0057
    },
    "f1_macro": {
     "folds": [
      0.9919,
      1.0,
      1.0,
      0.9926,
      1.0
     ],
     "mean": 0.9969,
     "std": 0.0043
    },
    "f1_weighted": {
     "folds": [
      0.9953,
      1.0,
      1.0,
      0.9976,
      1.0
     ],
     "mean": 0.9986,
     "std": 0.0021
    },
    "roc_auc_macro": {
     "folds": [
      1.0,
      1.0,
      1.0,
      1.0,
      1.0
     ],
     "mean": 1.0,
     "std": 0.0
    },
    "mcc": {
     "folds": [
      0.9925,
      1.0,
      1.0,
      0.9962,
      1.0
     ],
     "mean": 0.9978,
     "std": 0.0033
    },
    "cohen_kappa": {
     "folds": [
      0.9925,
      1.0,
      1.0,
      0.9962,
      1.0
     ],
     "mean": 0.9977,
     "std": 0.0034
    }
   },
   "stage2_multiclass": {
    "accuracy": {
     "folds": [
      0.6785,
      0.7629,
      0.7259,
      0.7536,
      0.7598
     ],
     "mean": 0.7362,
     "std": 0.0354
    },
    "precision_macro": {
     "folds": [
      0.6842,
      0.766,
      0.7436,
      0.7541,
      0.7559
     ],
     "mean": 0.7408,
     "std": 0.0326
    },
    "recall_macro": {
     "folds": [
      0.6841,
      0.7644,
      0.7281,
      0.7481,
      0.7613
     ],
     "mean": 0.7372,
     "std": 0.033
    },
    "f1_macro": {
     "folds": [
      0.683,
      0.7651,
      0.7343,
      0.7506,
      0.7571
     ],
     "mean": 0.738,
     "std": 0.0328
    },
    "f1_weighted": {
     "folds": [
      0.6763,
      0.763,
      0.7264,
      0.752,
      0.7571
     ],
     "mean": 0.735,
     "std": 0.0356
    },
    "roc_auc_macro": {
     "folds": [
      0.8612,
      0.9274,
      0.8934,
      0.9178,
      0.9228
     ],
     "mean": 0.9045,
     "std": 0.0275
    },
    "qwk": {
     "folds": [
      0.7117,
      0.8142,
      0.7328,
      0.7831,
      0.8091
     ],
     "mean": 0.7702,
     "std": 0.0459
    },
    "mcc": {
     "folds": [
      0.5591,
      0.6832,
      0.6254,
      0.6632,
      0.6806
     ],
     "mean": 0.6423,
     "std": 0.0519
    },
    "cohen_kappa": {
     "folds": [
      0.5584,
      0.6831,
      0.6247,
      0.6629,
      0.6797
     ],
     "mean": 0.6418,
     "std": 0.0521
    }
   },
   "stage2_binary": {
    "accuracy": {
     "folds": [
      0.8554,
      0.9025,
      0.8975,
      0.9129,
      0.9181
     ],
     "mean": 0.8973,
     "std": 0.0248
    },
    "precision_macro": {
     "folds": [
      0.8316,
      0.8785,
      0.8684,
      0.9069,
      0.9022
     ],
     "mean": 0.8775,
     "std": 0.0303
    },
    "recall_macro": {
     "folds": [
      0.8341,
      0.8385,
      0.8666,
      0.8876,
      0.8716
     ],
     "mean": 0.8597,
     "std": 0.0228
    },
    "f1_macro": {
     "folds": [
      0.8328,
      0.856,
      0.8675,
      0.8963,
      0.8854
     ],
     "mean": 0.8676,
     "std": 0.0249
    },
    "roc_auc": {
     "folds": [
      0.9105,
      0.9438,
      0.9476,
      0.961,
      0.9583
     ],
     "mean": 0.9442,
     "std": 0.0202
    },
    "mcc": {
     "folds": [
      0.6657,
      0.7159,
      0.7351,
      0.7942,
      0.7732
     ],
     "mean": 0.7368,
     "std": 0.0503
    },
    "cohen_kappa": {
     "folds": [
      0.6657,
      0.7125,
      0.7351,
      0.7927,
      0.7711
     ],
     "mean": 0.7354,
     "std": 0.0499
    }
   }
  },
  "test": {
   "stage1_hazard": {
    "precision_macro": 1.0,
    "recall_macro": 1.0,
    "f1_macro": 1.0,
    "precision_weighted": 1.0,
    "recall_weighted": 1.0,
    "f1_weighted": 1.0,
    "roc_auc_macro": 1.0,
    "roc_auc_weighted": 1.0,
    "roc_auc": 1.0,
    "mcc": 1.0,
    "cohen_kappa": 1.0
   },
   "stage2_multiclass_natural": {
    "precision_macro": 0.6428,
    "recall_macro": 0.7565,
    "f1_macro": 0.6839,
    "precision_weighted": 0.8485,
    "recall_weighted": 0.7884,
    "f1_weighted": 0.8074,
    "roc_auc_macro": 0.9351,
    "roc_auc_weighted": 0.9415,
    "roc_auc": 0.9351,
    "mcc": 0.6037,
    "cohen_kappa": 0.5856,
    "qwk": 0.7236
   },
   "stage2_binary": {
    "precision_macro": 0.7986,
    "recall_macro": 0.8701,
    "f1_macro": 0.8112,
    "precision_weighted": 0.8831,
    "recall_weighted": 0.831,
    "f1_weighted": 0.8399,
    "roc_auc": 0.95,
    "roc_auc_macro": 0.95,
    "roc_auc_weighted": 0.95,
    "mcc": 0.6649,
    "cohen_kappa": 0.631
   },
   "stage2_multiclass_capped": {
    "accuracy": 0.7724,
    "precision_macro": 0.7075,
    "recall_macro": 0.7561,
    "f1_macro": 0.7271,
    "precision_weighted": 0.7995,
    "recall_weighted": 0.7724,
    "f1_weighted": 0.7809,
    "roc_auc_macro": 0.9299,
    "roc_auc_weighted": 0.9331,
    "roc_auc": 0.9299,
    "mcc": 0.6505,
    "cohen_kappa": 0.6452,
    "qwk": 0.7734
   }
  },
  "models": "convnext_tiny"
 },
 "vit": {
  "cv": {
   "stage1_hazard": {
    "accuracy": {
     "folds": [
      1.0,
      1.0,
      0.993,
      0.9953,
      1.0
     ],
     "mean": 0.9977,
     "std": 0.0033
    },
    "precision_macro": {
     "folds": [
      1.0,
      1.0,
      0.9918,
      0.9972,
      1.0
     ],
     "mean": 0.9978,
     "std": 0.0036
    },
    "recall_macro": {
     "folds": [
      1.0,
      1.0,
      0.9876,
      0.9856,
      1.0
     ],
     "mean": 0.9946,
     "std": 0.0074
    },
    "f1_macro": {
     "folds": [
      1.0,
      1.0,
      0.9897,
      0.9912,
      1.0
     ],
     "mean": 0.9962,
     "std": 0.0053
    },
    "f1_weighted": {
     "folds": [
      1.0,
      1.0,
      0.9929,
      0.9953,
      1.0
     ],
     "mean": 0.9976,
     "std": 0.0033
    },
    "roc_auc_macro": {
     "folds": [
      1.0,
      1.0,
      0.9999,
      0.9987,
      1.0
     ],
     "mean": 0.9997,
     "std": 0.0006
    },
    "mcc": {
     "folds": [
      1.0,
      1.0,
      0.9887,
      0.9925,
      1.0
     ],
     "mean": 0.9962,
     "std": 0.0053
    },
    "cohen_kappa": {
     "folds": [
      1.0,
      1.0,
      0.9887,
      0.9925,
      1.0
     ],
     "mean": 0.9962,
     "std": 0.0053
    }
   },
   "stage2_multiclass": {
    "accuracy": {
     "folds": [
      0.679,
      0.7651,
      0.7273,
      0.7499,
      0.7492
     ],
     "mean": 0.7341,
     "std": 0.0336
    },
    "precision_macro": {
     "folds": [
      0.6919,
      0.768,
      0.7398,
      0.7522,
      0.7512
     ],
     "mean": 0.7406,
     "std": 0.029
    },
    "recall_macro": {
     "folds": [
      0.6767,
      0.7685,
      0.7339,
      0.7508,
      0.7496
     ],
     "mean": 0.7359,
     "std": 0.0353
    },
    "f1_macro": {
     "folds": [
      0.6825,
      0.7672,
      0.7347,
      0.7507,
      0.7501
     ],
     "mean": 0.737,
     "std": 0.0326
    },
    "f1_weighted": {
     "folds": [
      0.6769,
      0.7651,
      0.7294,
      0.7504,
      0.7505
     ],
     "mean": 0.7344,
     "std": 0.0346
    },
    "roc_auc_macro": {
     "folds": [
      0.8751,
      0.9228,
      0.9028,
      0.9177,
      0.9216
     ],
     "mean": 0.908,
     "std": 0.02
    },
    "qwk": {
     "folds": [
      0.6958,
      0.8059,
      0.7129,
      0.7884,
      0.8071
     ],
     "mean": 0.762,
     "std": 0.0535
    },
    "mcc": {
     "folds": [
      0.5582,
      0.6872,
      0.6293,
      0.6598,
      0.6657
     ],
     "mean": 0.64,
     "std": 0.0502
    },
    "cohen_kappa": {
     "folds": [
      0.5573,
      0.6866,
      0.6277,
      0.6591,
      0.6654
     ],
     "mean": 0.6392,
     "std": 0.0504
    }
   },
   "stage2_binary": {
    "accuracy": {
     "folds": [
      0.8556,
      0.8897,
      0.89,
      0.9115,
      0.9134
     ],
     "mean": 0.892,
     "std": 0.0233
    },
    "precision_macro": {
     "folds": [
      0.8323,
      0.8429,
      0.8875,
      0.9032,
      0.8941
     ],
     "mean": 0.872,
     "std": 0.0321
    },
    "recall_macro": {
     "folds": [
      0.8328,
      0.8532,
      0.8195,
      0.8884,
      0.8671
     ],
     "mean": 0.8522,
     "std": 0.0273
    },
    "f1_macro": {
     "folds": [
      0.8325,
      0.8479,
      0.8454,
      0.8952,
      0.8794
     ],
     "mean": 0.8601,
     "std": 0.0261
    },
    "roc_auc": {
     "folds": [
      0.9187,
      0.9365,
      0.9449,
      0.9575,
      0.9531
     ],
     "mean": 0.9421,
     "std": 0.0154
    },
    "mcc": {
     "folds": [
      0.665,
      0.696,
      0.7037,
      0.7914,
      0.7607
     ],
     "mean": 0.7234,
     "std": 0.0514
    },
    "cohen_kappa": {
     "folds": [
      0.665,
      0.6958,
      0.6926,
      0.7905,
      0.759
     ],
     "mean": 0.7206,
     "std": 0.0521
    }
   }
  },
  "test": {
   "stage1_hazard": {
    "precision_macro": 0.999,
    "recall_macro": 0.9988,
    "f1_macro": 0.9989,
    "precision_weighted": 0.9981,
    "recall_weighted": 0.9981,
    "f1_weighted": 0.9981,
    "roc_auc_macro": 1.0,
    "roc_auc_weighted": 1.0,
    "roc_auc": 1.0,
    "mcc": 0.997,
    "cohen_kappa": 0.997
   },
   "stage2_multiclass_natural": {
    "precision_macro": 0.6466,
    "recall_macro": 0.7495,
    "f1_macro": 0.6861,
    "precision_weighted": 0.8444,
    "recall_weighted": 0.796,
    "f1_weighted": 0.8122,
    "roc_auc_macro": 0.9345,
    "roc_auc_weighted": 0.9383,
    "roc_auc": 0.9345,
    "mcc": 0.6048,
    "cohen_kappa": 0.5913,
    "qwk": 0.7249
   },
   "stage2_binary": {
    "precision_macro": 0.7997,
    "recall_macro": 0.8681,
    "f1_macro": 0.8139,
    "precision_weighted": 0.8809,
    "recall_weighted": 0.8349,
    "f1_weighted": 0.8431,
    "roc_auc": 0.9451,
    "roc_auc_macro": 0.9451,
    "roc_auc_weighted": 0.9451,
    "mcc": 0.6644,
    "cohen_kappa": 0.6351
   },
   "stage2_multiclass_capped": {
    "accuracy": 0.7731,
    "precision_macro": 0.7069,
    "recall_macro": 0.7487,
    "f1_macro": 0.7248,
    "precision_weighted": 0.7921,
    "recall_weighted": 0.7731,
    "f1_weighted": 0.7796,
    "roc_auc_macro": 0.9289,
    "roc_auc_weighted": 0.9304,
    "roc_auc": 0.9289,
    "mcc": 0.6455,
    "cohen_kappa": 0.6424,
    "qwk": 0.7674
   }
  },
  "models": "vit_base_patch16_224"
 },
 "checksums": {
  "stage1": "cb0de2a1361950bf",
  "stage2": "e051b66afeb3dc37"
 }
}''')
BASE_NAMES = {'convnext': 'Existing ConvNeXt (convnext_tiny siamese)',
              'vit': 'Existing ViT (vit_base_patch16_224 siamese)'}
ENS_NAME  = 'NEW ConvNeXt+ViT trained fusion ensemble (proposed)'
SOFT_NAME = 'ABLATION: ConvNeXt+ViT probability soft-voting (baseline, NOT proposed)'
CMP_METRICS = ['accuracy', 'f1_macro', 'roc_auc', 'mcc', 'cohen_kappa', 'qwk']
CV_VIEWS = ['stage1_hazard', 'stage2_multiclass', 'stage2_binary']
TEST_VIEWS = ['stage1_hazard', 'stage2_multiclass_natural', 'stage2_multiclass_capped', 'stage2_binary']

def find_results_dir(name):
    for root in ['/kaggle/input', '/kaggle/working']:
        if os.path.isdir(root):
            for dp, dn, _ in os.walk(root):
                if os.path.basename(dp) == name and os.path.isdir(os.path.join(dp, 'stage1_hazard')):
                    return dp
    return None

BASE_DIRS = {t: find_results_dir(f"{t}_results") for t in ('convnext', 'vit')}
print("existing result folders found:", BASE_DIRS)

def _roc(m):
    return m.get('roc_auc_macro', m.get('roc_auc', np.nan))

def baseline_cv(tag):
    """{view: {metric: [5 fold values]}} from CSVs if attached, else embedded."""
    out, src = {}, 'embedded (printed in the existing notebook)'
    bd = BASE_DIRS.get(tag)
    files = {'stage1_hazard': 'stage1_fold_metrics.csv', 'stage2_multiclass': 'stage2_multiclass_fold_metrics.csv',
             'stage2_binary': 'stage2_binary_fold_metrics.csv'}
    if bd and all(os.path.exists(os.path.join(bd, f)) for f in files.values()):
        src = bd
        for v, fn in files.items():
            df = pd.read_csv(os.path.join(bd, fn))
            out[v] = {k: df[k].tolist() for k in df.columns if k != 'fold'}
            out[v]['roc_auc'] = (df['roc_auc_macro'] if 'roc_auc_macro' in df else df['roc_auc']).tolist()
    else:
        for v in CV_VIEWS:
            tab = EMBEDDED_BASELINES[tag]['cv'][v]
            out[v] = {k: tab[k]['folds'] for k in tab}
            out[v]['roc_auc'] = (tab.get('roc_auc_macro') or tab.get('roc_auc'))['folds']
    return out, src

def baseline_test(tag):
    bd = BASE_DIRS.get(tag)
    p = os.path.join(bd, 'final_test_metrics.csv') if bd else None
    if p and os.path.exists(p):
        df = pd.read_csv(p).set_index('stage')
        return {v: df.loc[v].to_dict() for v in TEST_VIEWS if v in df.index}, bd
    return EMBEDDED_BASELINES[tag]['test'], 'embedded (printed in the existing notebook)'

ens_cv = {'stage1_hazard': s1_fold_metrics, 'stage2_multiclass': s2_fold_metrics, 'stage2_binary': s2b_fold_metrics}
ens_test = {'stage1_hazard': t1, 'stage2_multiclass_natural': t2, 'stage2_multiclass_capped': t2c, 'stage2_binary': t2b}

cv_rows, fold_rows, test_rows, sources = [], [], [], {}
def add_cv(model_name, view, getter, n_folds=N_SPLITS):
    for mk in CMP_METRICS:
        vals = getter(mk)
        if vals is None: continue
        vals = [float(v) for v in vals]
        cv_rows.append({'model': model_name, 'view': view, 'metric': mk, 'mean': np.mean(vals),
                        'std': np.std(vals, ddof=1) if len(vals) > 1 else 0.0,
                        'mean ± std': f"{np.mean(vals):.4f} ± {np.std(vals, ddof=1) if len(vals) > 1 else 0:.4f}"})
        fold_rows.append({'model': model_name, 'view': view, 'metric': mk,
                          **{f'fold_{i+1}': v for i, v in enumerate(vals)}})

for tag in ('convnext', 'vit'):
    bcv, src = baseline_cv(tag); sources[f'{tag}_cv'] = src
    for v in CV_VIEWS:
        add_cv(BASE_NAMES[tag], v, lambda mk, v=v, b=bcv: b[v].get(mk) if mk in b[v] else None)
    bt, src = baseline_test(tag); sources[f'{tag}_test'] = src
    for v in TEST_VIEWS:
        if v in bt:
            test_rows.append({'model': BASE_NAMES[tag], 'view': v,
                              **{mk: (_roc(bt[v]) if mk == 'roc_auc' else bt[v].get(mk, np.nan)) for mk in CMP_METRICS}})
for v in CV_VIEWS:
    add_cv(ENS_NAME, v, lambda mk, v=v: None if (mk == 'qwk' and v != 'stage2_multiclass')
           else [(_roc(m) if mk == 'roc_auc' else m[mk]) for m in ens_cv[v]])
for v in TEST_VIEWS:
    test_rows.append({'model': ENS_NAME, 'view': v,
                      **{mk: (_roc(ens_test[v]) if mk == 'roc_auc' else ens_test[v].get(mk, np.nan)) for mk in CMP_METRICS}})

# ---------------- ABLATION: soft voting of the existing models' probabilities ----------------
SOFT_VOTING_DONE = False
def _load_preds(bd, rel):
    p = os.path.join(bd, rel)
    return pd.read_csv(p) if os.path.exists(p) else None

if all(BASE_DIRS.values()):
    try:
        cb, vb = BASE_DIRS['convnext'], BASE_DIRS['vit']
        sv_cv = {v: [] for v in CV_VIEWS}
        for f in range(N_SPLITS):
            for view, sub, key, names, ordinal in [('stage1_hazard', 'stage1_hazard', 'image_id', HAZARD_NAMES, False),
                                                   ('stage2_multiclass', 'stage2_severity_multiclass', 'crop_id', DAMAGE_NAMES, True)]:
                a = _load_preds(cb, f"{sub}/fold_{f+1}/predictions.csv"); b = _load_preds(vb, f"{sub}/fold_{f+1}/predictions.csv")
                mg = a.merge(b, on=key, suffixes=('_c', '_v'))
                assert len(mg) == len(a) == len(b), "existing fold predictions are not aligned"
                pc = mg[[f'prob_{n}_c' for n in names]].values; pv = mg[[f'prob_{n}_v' for n in names]].values
                pr = (pc + pv) / 2; y = mg['ground_truth_c'].map({n: i for i, n in enumerate(names)}).values
                sv_cv[view].append(compute_metrics(y, pr.argmax(1), pr, 4, ordinal=ordinal))
                if view == 'stage2_multiclass':
                    yb, ybp, pb = to_binary(y, pr)
                    sv_cv['stage2_binary'].append(compute_metrics(yb, ybp, pb, 2))
        for v in CV_VIEWS:
            add_cv(SOFT_NAME, v, lambda mk, v=v: None if (mk == 'qwk' and v != 'stage2_multiclass')
                   else [(_roc(m) if mk == 'roc_auc' else m[mk]) for m in sv_cv[v]])
        a = _load_preds(cb, "final_test/stage1_predictions.csv"); b = _load_preds(vb, "final_test/stage1_predictions.csv")
        mg = a.merge(b, on='image_id', suffixes=('_c', '_v'))
        pr = (mg[[f'prob_{n}_c' for n in HAZARD_NAMES]].values + mg[[f'prob_{n}_v' for n in HAZARD_NAMES]].values) / 2
        y = mg['ground_truth_c'].map(hazard_to_idx).values
        sv_t = {'stage1_hazard': compute_metrics(y, pr.argmax(1), pr, 4)}
        a = _load_preds(cb, "final_test/stage2_predictions.csv"); b = _load_preds(vb, "final_test/stage2_predictions.csv")
        mg = a.merge(b, on='crop_id', suffixes=('_c', '_v'))
        pr = (mg[[f'prob_{n}_c' for n in DAMAGE_NAMES]].values + mg[[f'prob_{n}_v' for n in DAMAGE_NAMES]].values) / 2
        y = mg['ground_truth_c'].map(damage_to_idx).values
        sv_t['stage2_multiclass_natural'] = compute_metrics(y, pr.argmax(1), pr, 4, ordinal=True)
        cm_ = mg['crop_id'].isin(set(capped['crop_id'])).values
        sv_t['stage2_multiclass_capped'] = compute_metrics(y[cm_], pr[cm_].argmax(1), pr[cm_], 4, ordinal=True)
        yb, ybp, pb = to_binary(y, pr); sv_t['stage2_binary'] = compute_metrics(yb, ybp, pb, 2)
        for v in TEST_VIEWS:
            test_rows.append({'model': SOFT_NAME, 'view': v,
                              **{mk: (_roc(sv_t[v]) if mk == 'roc_auc' else sv_t[v].get(mk, np.nan)) for mk in CMP_METRICS}})
        SOFT_VOTING_DONE = True
        print("soft-voting ablation computed from the existing prediction files")
    except Exception as e:
        print("soft-voting ablation skipped:", repr(e))
else:
    print("soft-voting ablation skipped: attach the existing convnext_results/ and vit_results/ folders to enable it")

cv_cmp = pd.DataFrame(cv_rows); fold_cmp = pd.DataFrame(fold_rows); test_cmp = pd.DataFrame(test_rows)
cv_cmp.to_csv(os.path.join(D['cmp'], "cv_comparison_long.csv"), index=False)
cv_wide = cv_cmp.pivot_table(index=['view', 'model'], columns='metric', values='mean ± std', aggfunc='first')
cv_wide = cv_wide[[m for m in CMP_METRICS if m in cv_wide.columns]]
cv_wide.to_csv(os.path.join(D['cmp'], "cv_comparison_mean_std.csv"))
fold_cmp.to_csv(os.path.join(D['cmp'], "cv_foldwise_all_models.csv"), index=False)
fold_cmp[fold_cmp['model'] == ENS_NAME].to_csv(os.path.join(D['cmp'], "cv_foldwise_ensemble.csv"), index=False)
test_cmp.to_csv(os.path.join(D['cmp'], "final_test_comparison.csv"), index=False)
jdump({'sources': sources, 'fold_checksums': FOLD_CHECKSUMS,
       'fold_matched_with_baselines': {'stage1': S1_MATCHES_BASELINE, 'stage2': S2_MATCHES_BASELINE},
       'soft_voting_ablation_computed': SOFT_VOTING_DONE,
       'note': 'baselines were not retrained; soft voting is an ablation baseline, not the proposed model',
       'cv': cv_cmp.to_dict('records'), 'test': test_cmp.to_dict('records')},
      os.path.join(D['cmp'], "comparison.json"))

pd.set_option('display.width', 220)
print("\n5-FOLD CV  (mean ± std)"); print(cv_wide.to_string())
print("\nFOLD-WISE - NEW ENSEMBLE"); print(fold_cmp[fold_cmp['model'] == ENS_NAME].round(4).to_string(index=False))
print("\nFINAL HELD-OUT TEST"); print(test_cmp.round(4).to_string(index=False))
print("\nsources:", json.dumps(sources, indent=1))

# bar chart: macro-F1 and QWK CV mean ± std
fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
for ax, v in zip(axes, CV_VIEWS):
    sub = cv_cmp[(cv_cmp['view'] == v) & (cv_cmp['metric'] == 'f1_macro')]
    ax.bar(range(len(sub)), sub['mean'], yerr=sub['std'], capsize=4,
           color=['#999', '#bbb', '#1f77b4', '#ddd'][:len(sub)])
    ax.set_xticks(range(len(sub)))
    ax.set_xticklabels([m.split('(')[0].replace('Existing ', '').replace('ABLATION: ', 'Ablation\n')
                        .replace('NEW ', 'NEW\n') for m in sub['model']], fontsize=8)
    lo = max(0, sub['mean'].min() - 3 * sub['std'].max() - 0.02)
    ax.set_ylim(lo, min(1.0, sub['mean'].max() + 3 * sub['std'].max() + 0.02))
    ax.set_title(f"{v} - CV macro-F1 (mean ± std)"); ax.grid(alpha=.3, axis='y')
plt.tight_layout(); plt.savefig(os.path.join(D['cmp'], "cv_macro_f1_comparison.png"), dpi=150); plt.show(); plt.close(fig)

---
## 13. Deployment bundle (for the frontend phase)

`deployment_bundle/` is self-contained: final ensemble checkpoints + JSON configs + `inference.py` exposing `load_models()`, `predict_hazard(pre, post)`, `predict_damage(pre, post)`, `predict_full(pre, post, buildings=None)`. The bundle is then **imported and executed** below to produce `sample_prediction.json`, and its predictions are cross-checked against the notebook's own test predictions.

In [ ]:
INFERENCE_PY = r'''"""
Disaster XAI-Net - ConvNeXt + ViT trained feature-fusion ensemble: inference API.

    from inference import load_models, predict_hazard, predict_damage, predict_full
    load_models()                                   # loads both stages from this bundle
    predict_hazard(pre, post)                       # full tile pair  -> hazard
    predict_damage(pre_crop, post_crop)             # one building crop pair -> severity
    predict_full(pre, post, buildings=[...])        # tile pair + building polygons/bboxes

Images may be file paths, PIL images, HxWx3 uint8 numpy arrays, or raw bytes.
All coordinates are IMAGE PIXELS of the full-resolution tile (x right, y down).
No geographic coordinates are produced; lat/lon is passed through only if the caller supplies it.
"""
import io
import os
import json
import re

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from PIL import Image

BUNDLE_DIR = os.path.dirname(os.path.abspath(__file__))
_STATE = {}


# ----------------------------------------------------------------- architecture
class SiameseBranch(nn.Module):
    def __init__(self, model_name, img_size=None, use_pair=True, pretrained=False):
        super().__init__()
        kw = dict(pretrained=pretrained, num_classes=0)
        if img_size is not None and any(t in model_name for t in ("vit", "deit", "beit")):
            kw["img_size"] = img_size
        self.backbone = timm.create_model(model_name, **kw)
        self.use_pair = use_pair
        self.img_feat_dim = self.backbone.num_features
        self.feat_dim = self.img_feat_dim * (2 if use_pair else 1)

    def forward(self, pre_img, post_img):
        f_post = self.backbone(post_img)
        if self.use_pair:
            return torch.cat([self.backbone(pre_img), f_post], dim=1)
        return f_post


class ConvNeXtViTFusion(nn.Module):
    def __init__(self, num_classes, conv_model, vit_model, vit_img_size=None, use_pair=True,
                 hidden=512, dropout=0.3, pretrained=False):
        super().__init__()
        self.conv = SiameseBranch(conv_model, None, use_pair, pretrained)
        self.vit = SiameseBranch(vit_model, vit_img_size, use_pair, pretrained)
        self.conv_norm = nn.LayerNorm(self.conv.feat_dim)
        self.vit_norm = nn.LayerNorm(self.vit.feat_dim)
        self.fused_dim = self.conv.feat_dim + self.vit.feat_dim
        self.fusion = nn.Sequential(nn.Dropout(dropout), nn.Linear(self.fused_dim, hidden), nn.ReLU(),
                                    nn.Dropout(dropout), nn.Linear(hidden, num_classes))

    def forward(self, pre_img, post_img):
        fc, fv = self.conv(pre_img, post_img), self.vit(pre_img, post_img)
        return self.fusion(torch.cat([self.conv_norm(fc), self.vit_norm(fv)], dim=1))


# ----------------------------------------------------------------- loading
def _json(name, bundle_dir):
    with open(os.path.join(bundle_dir, name)) as fh:
        return json.load(fh)


def _torch_load(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def load_models(bundle_dir=None, device=None):
    """Load both final ensemble checkpoints. Safe to call repeatedly (cached)."""
    bundle_dir = bundle_dir or BUNDLE_DIR
    if _STATE.get("bundle_dir") == bundle_dir and "models" in _STATE:
        return _STATE
    device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
    cfg = _json("model_config.json", bundle_dir)
    pre = _json("preprocessing_config.json", bundle_dir)
    labels = _json("label_maps.json", bundle_dir)
    models = {}
    for stage in ("stage1", "stage2"):
        ck = _torch_load(os.path.join(bundle_dir, cfg["checkpoints"][stage]))
        a = ck["meta"]["architecture"]
        m = ConvNeXtViTFusion(a["num_classes"], a["conv_model"], a["vit_model"], a["vit_img_size"],
                              a["use_pair"], a["fusion_hidden"], a["dropout"], pretrained=False)
        m.load_state_dict({k: (v.float() if v.is_floating_point() else v) for k, v in ck["state_dict"].items()})
        models[stage] = m.to(device).eval()
    _STATE.update(bundle_dir=bundle_dir, device=device, models=models, cfg=cfg, pre=pre,
                  hazard_names=labels["hazard"]["index_to_label"],
                  damage_names=labels["damage"]["index_to_label"],
                  mean=torch.tensor(pre["normalization"]["mean"]).view(3, 1, 1),
                  std=torch.tensor(pre["normalization"]["std"]).view(3, 1, 1))
    return _STATE


def _ensure():
    return _STATE if "models" in _STATE else load_models()


# ----------------------------------------------------------------- preprocessing
def _to_pil(x):
    if isinstance(x, Image.Image):
        return x.convert("RGB")
    if isinstance(x, (bytes, bytearray)):
        return Image.open(io.BytesIO(x)).convert("RGB")
    if isinstance(x, (str, os.PathLike)):
        return Image.open(x).convert("RGB")
    if isinstance(x, np.ndarray):
        a = x
        if a.dtype != np.uint8:
            a = (np.clip(a, 0, 1) * 255).astype(np.uint8) if a.max() <= 1.0 else a.astype(np.uint8)
        if a.ndim == 2:
            a = np.stack([a] * 3, -1)
        return Image.fromarray(a[..., :3]).convert("RGB")
    raise TypeError(f"unsupported image type {type(x)}")


def _norm(arr):
    s = _STATE
    t = torch.from_numpy(np.ascontiguousarray(arr)).permute(2, 0, 1).float().div_(255)
    return (t - s["mean"]) / s["std"]


def _prep_tile(img):
    """Stage 1 eval transform: Resize(256,256) -> CenterCrop(224) -> normalise."""
    s1 = _STATE["pre"]["stage1"]
    size = int(s1["image_size"])
    im = img.resize((256, 256), Image.BILINEAR)
    off = (256 - size) // 2
    im = im.crop((off, off, off + size, off + size))
    return _norm(np.asarray(im))


def _prep_crop(img):
    """Stage 2: building crop resized to 64x64 bilinear -> normalise (no augmentation)."""
    c = int(_STATE["pre"]["stage2"]["crop"]["resize"][0])
    return _norm(np.asarray(img.resize((c, c), Image.BILINEAR)))


def padded_box(x0, y0, x1, y1, W, H, pad=None):
    pad = _STATE["pre"]["stage2"]["crop"]["pad_fraction"] if pad is None else pad
    w, h = x1 - x0, y1 - y0
    px, py = w * pad, h * pad
    bx0, by0 = max(0, x0 - px), max(0, y0 - py)
    bx1, by1 = min(W, x1 + px), min(H, y1 + py)
    if bx1 - bx0 < 8:
        bx0, bx1 = max(0, bx0 - 4), min(W, bx1 + 4)
    if by1 - by0 < 8:
        by0, by1 = max(0, by0 - 4), min(H, by1 + 4)
    return (int(bx0), int(by0), int(max(bx0 + 1, bx1)), int(max(by0 + 1, by1)))


def _parse_wkt(wkt):
    m = re.search(r"\(\((.*?)\)\)", wkt or "")
    if not m:
        return None
    return [[float(v) for v in c.strip().split()[:2]] for c in m.group(1).split(",")]


def _building_geometry(b):
    """bbox (x0,y0,x1,y1) from 'bbox' | 'polygon' [[x,y],...] | 'wkt' (image pixels)."""
    poly = b.get("polygon")
    if poly is None and b.get("wkt"):
        poly = _parse_wkt(b["wkt"])
    if b.get("bbox") is not None:
        x0, y0, x1, y1 = [float(v) for v in b["bbox"]]
    elif poly:
        p = np.asarray(poly, dtype=float)
        x0, y0 = p.min(0)
        x1, y1 = p.max(0)
    else:
        raise ValueError("each building needs 'bbox', 'polygon' or 'wkt' in image-pixel coordinates")
    return (x0, y0, x1, y1), poly


# ----------------------------------------------------------------- outputs
def _probs_dict(p, names):
    return {n: float(p[i]) for i, n in enumerate(names)}


def _damage_record(p):
    names = _STATE["damage_names"]
    k = int(np.argmax(p))
    p_dmg = float(p[1] + p[2] + p[3])
    return {"severity": names[k], "severity_index": k, "confidence": float(p[k]),
            "probabilities": _probs_dict(p, names),
            "p_damaged": p_dmg, "damaged": bool(p_dmg > 0.5)}


# ----------------------------------------------------------------- public API
@torch.no_grad()
def predict_hazard(pre, post):
    """Stage 1 on a full pre/post tile pair."""
    s = _ensure()
    a, b = _prep_tile(_to_pil(pre)), _prep_tile(_to_pil(post))
    logits = s["models"]["stage1"](a[None].to(s["device"]), b[None].to(s["device"]))
    p = F.softmax(logits.float(), 1)[0].cpu().numpy()
    k = int(np.argmax(p))
    names = s["hazard_names"]
    return {"hazard": names[k], "hazard_index": k, "confidence": float(p[k]),
            "probabilities": _probs_dict(p, names)}


@torch.no_grad()
def predict_damage(pre, post):
    """Stage 2 on ONE building crop pair (already cropped around the building)."""
    s = _ensure()
    a, b = _prep_crop(_to_pil(pre)), _prep_crop(_to_pil(post))
    logits = s["models"]["stage2"](a[None].to(s["device"]), b[None].to(s["device"]))
    return _damage_record(F.softmax(logits.float(), 1)[0].cpu().numpy())


@torch.no_grad()
def predict_full(pre, post, buildings=None, batch_size=256):
    """Stage 1 on the tile + Stage 2 on every building.

    buildings: list of dicts, each with 'bbox' [x0,y0,x1,y1] OR 'polygon' [[x,y],...] OR 'wkt'
               ("POLYGON ((x y, ...))") in full-tile image pixels; optional 'building_id', 'uid',
               'lat', 'lon' (passed through untouched, never estimated).
               If None, the whole tile is scored as a single region (region_type='whole_image').
    """
    s = _ensure()
    pre_im, post_im = _to_pil(pre), _to_pil(post)
    W, H = post_im.size
    hazard = predict_hazard(pre_im, post_im)
    whole = buildings is None
    if whole:
        buildings = [{"building_id": "whole_image", "bbox": [0, 0, W, H]}]
    recs, xs_pre, xs_post = [], [], []
    for i, b in enumerate(buildings):
        (x0, y0, x1, y1), poly = _building_geometry(b)
        box = (0, 0, W, H) if whole else padded_box(x0, y0, x1, y1, W, H)
        xs_pre.append(_prep_crop(pre_im.crop(box)))
        xs_post.append(_prep_crop(post_im.crop(box)))
        recs.append({"building_id": b.get("building_id", f"building_{i}"), "uid": b.get("uid"),
                     "region_type": "whole_image" if whole else "building",
                     "bbox_xyxy": [float(x0), float(y0), float(x1), float(y1)],
                     "crop_box_xyxy": list(map(int, box)),
                     "polygon_xy": poly,
                     "lat": b.get("lat"), "lon": b.get("lon")})
    probs = []
    for j in range(0, len(recs), batch_size):
        a = torch.stack(xs_pre[j:j + batch_size]).to(s["device"])
        c = torch.stack(xs_post[j:j + batch_size]).to(s["device"])
        probs.append(F.softmax(s["models"]["stage2"](a, c).float(), 1).cpu().numpy())
    probs = np.concatenate(probs) if probs else np.zeros((0, 4))
    for r, p in zip(recs, probs):
        r.update(_damage_record(p))
    names = s["damage_names"]
    counts = {n: int(sum(r["severity"] == n for r in recs)) for n in names}
    return {
        "model": {"name": s["cfg"]["model_name"], "version": s["cfg"]["model_version"]},
        "coordinate_system": "image_pixels (x right, y down) of the full-resolution post tile",
        "image": {"width": W, "height": H},
        "hazard": hazard,
        "buildings": recs,
        "summary": {"n_buildings": len(recs), "severity_counts": counts,
                    "n_damaged": int(sum(r["damaged"] for r in recs)),
                    "mean_p_damaged": float(np.mean([r["p_damaged"] for r in recs])) if recs else None},
    }


if __name__ == "__main__":
    import argparse
    ap = argparse.ArgumentParser(description="Disaster XAI-Net ensemble inference")
    ap.add_argument("pre"); ap.add_argument("post")
    ap.add_argument("--labels", help="xBD-style *_post_disaster.json with features.xy polygons")
    args = ap.parse_args()
    blds = None
    if args.labels:
        with open(args.labels) as fh:
            js = json.load(fh)
        blds = [{"building_id": f"b{i}", "uid": f.get("properties", {}).get("uid"), "wkt": f["wkt"]}
                for i, f in enumerate(js.get("features", {}).get("xy", [])) if _parse_wkt(f.get("wkt"))]
    print(json.dumps(predict_full(args.pre, args.post, blds), indent=2))
'''

B = D['bundle']
os.makedirs(os.path.join(B, 'checkpoints'), exist_ok=True)
BUNDLE_CKPTS = {'stage1': 'checkpoints/stage1_ensemble_final.pth',
                'stage2': 'checkpoints/stage2_ensemble_final.pth'}
for st, src in [('stage1', S1_FINAL_PATH), ('stage2', S2_FINAL_PATH)]:
    dst = os.path.join(B, BUNDLE_CKPTS[st])
    if os.path.exists(dst): os.remove(dst)
    try:
        os.link(src, dst)                 # hard link: no extra disk inside /kaggle/working
    except OSError:
        shutil.copy2(src, dst)

with open(os.path.join(B, 'inference.py'), 'w') as fh:
    fh.write(INFERENCE_PY)

MODEL_VERSION = f"{ARCH_TAG}-{datetime.datetime.now():%Y%m%d}"
model_config = {
    'model_name': MODEL_NAME, 'model_version': MODEL_VERSION, 'arch_tag': ARCH_TAG,
    'checkpoints': BUNDLE_CKPTS,
    'stages': {
        'stage1': {'task': 'hazard classification', 'input': 'pre/post full tile pair',
                   'num_classes': 4, 'input_size': S1_IMG, 'vit_img_size': None},
        'stage2': {'task': 'building damage severity', 'input': 'pre/post building crop pair',
                   'num_classes': 4, 'input_size': S2_CROP, 'vit_img_size': S2_CROP,
                   'derived_binary': 'damaged = minor+major+destroyed; P(damaged)=P(minor)+P(major)+P(destroyed); threshold 0.5'}},
    'device': 'cuda if available else cpu', 'precision': 'float32',
}
architecture_config = {
    'name': MODEL_NAME, 'type': 'trained feature-fusion ensemble (NOT probability averaging)',
    'branches': {
        'convnext': {'timm_model': CONV_MODEL, 'siamese': True, 'shared_weights_pre_post': True,
                     'per_image_features': FEATURE_DIMS['stage1']['convnext_per_image'],
                     'branch_vector': 'concat(f_pre, f_post)', 'branch_dim': FEATURE_DIMS['stage1']['convnext_branch'],
                     'post_image_forwarded_first': True,
                     'gradcam_layers': {'stage1': 'conv.backbone.stages[3]', 'stage2': 'conv.backbone.stages[1]'}},
        'vit': {'timm_model': VIT_MODEL, 'siamese': True, 'shared_weights_pre_post': True,
                'per_image_features': FEATURE_DIMS['stage1']['vit_per_image'],
                'branch_vector': 'concat(f_pre, f_post)', 'branch_dim': FEATURE_DIMS['stage1']['vit_branch'],
                'img_size': {'stage1': S1_IMG, 'stage2': S2_CROP},
                'gradcam_layer': 'vit.backbone.blocks[-1].norm1 (tokens reshaped to patch grid)'}},
    'fusion': {'branch_norm': 'LayerNorm per branch', 'concat_dim': FEATURE_DIMS['stage1']['fused'],
               'mlp': f"Dropout({DROPOUT}) -> Linear({FEATURE_DIMS['stage1']['fused']},{FUSION_HIDDEN}) -> ReLU -> "
                      f"Dropout({DROPOUT}) -> Linear({FUSION_HIDDEN},4)"},
    'training': {'joint_fine_tuning': True, 'lr': {'convnext_backbone': LR_CONV, 'vit_backbone': LR_VIT,
                 'fusion_head': LR_HEAD}, 'optimizer': 'AdamW', 'weight_decay': WEIGHT_DECAY,
                 'scheduler': 'cosine', 'loss': 'class-weighted CE + label smoothing 0.05'},
    'feature_dimensions': FEATURE_DIMS,
}
label_maps = {
    'hazard': {'index_to_label': HAZARD_NAMES, 'label_to_index': hazard_to_idx},
    'damage': {'index_to_label': DAMAGE_NAMES, 'label_to_index': damage_to_idx, 'ordinal': True},
    'binary': {'index_to_label': BINARY_NAMES, 'damaged_from': ['minor-damage', 'major-damage', 'destroyed']},
}
bundle_pre = copy.deepcopy(PREPROCESSING_CONFIG)
bundle_pre['stage1'].update({'image_size': S1_IMG, 'resize': [256, 256], 'center_crop': S1_IMG})
inference_schema = {
    'coordinate_system': 'image pixels of the full-resolution tile, origin top-left, x right, y down',
    'geo': 'no latitude/longitude is ever computed; lat/lon fields are pass-through or null',
    'functions': {
        'load_models(bundle_dir=None, device=None)': 'loads both stages; cached',
        'predict_hazard(pre, post)': {'input': 'full pre/post tile (path | PIL | HxWx3 uint8 | bytes)',
            'output': {'hazard': 'str', 'hazard_index': 'int', 'confidence': 'float',
                       'probabilities': {n: 'float' for n in HAZARD_NAMES}}},
        'predict_damage(pre, post)': {'input': 'one building crop pair (already cropped)',
            'output': {'severity': 'str', 'severity_index': 'int', 'confidence': 'float',
                       'probabilities': {n: 'float' for n in DAMAGE_NAMES},
                       'p_damaged': 'float = P(minor)+P(major)+P(destroyed)', 'damaged': 'bool (p_damaged > 0.5)'}},
        'predict_full(pre, post, buildings=None)': {
            'input': {'pre': 'tile', 'post': 'tile', 'buildings': "list of {building_id?, uid?, bbox:[x0,y0,x1,y1] | "
                      "polygon:[[x,y],...] | wkt:'POLYGON ((x y, ...))', lat?, lon?} or null (whole tile)"},
            'output': {'model': {'name': 'str', 'version': 'str'}, 'coordinate_system': 'str',
                       'image': {'width': 'int', 'height': 'int'}, 'hazard': 'predict_hazard output',
                       'buildings': [{'building_id': 'str', 'uid': 'str|null', 'region_type': 'building|whole_image',
                                      'bbox_xyxy': '[float]*4', 'crop_box_xyxy': '[int]*4 (25% padded)',
                                      'polygon_xy': '[[x,y],...]|null', 'lat': 'float|null', 'lon': 'float|null',
                                      '...': 'predict_damage output fields'}],
                       'summary': {'n_buildings': 'int', 'severity_counts': {n: 'int' for n in DAMAGE_NAMES},
                                   'n_damaged': 'int', 'mean_p_damaged': 'float|null'}}}},
}
model_metadata = {
    'model_name': MODEL_NAME, 'model_version': MODEL_VERSION,
    'created': datetime.datetime.now().isoformat(timespec='seconds'),
    'dataset': 'Kaggle lubdham/disaster-separate-data (hazard_data/<hazard>/{train,test})',
    'n_dev_pairs': len(dev_df), 'n_test_pairs': len(test_df), 'n_dev_crops': len(sel), 'n_test_crops': len(test_sel),
    'fold_checksums': FOLD_CHECKSUMS, 'seed': SEED, 'n_splits': N_SPLITS,
    'final_epochs': {'stage1': S1_FINAL_EPOCHS, 'stage2': S2_FINAL_EPOCHS},
    'cv_summary': {'stage1': s1_summary.to_dict('records'), 'stage2_multiclass': s2_summary.to_dict('records'),
                   'stage2_binary': s2b_summary.to_dict('records')},
    'final_test_metrics': final_test_metrics.to_dict('records'),
    'checkpoint_meta': {'stage1': {k: v for k, v in S1_FINAL_META.items() if k != 'history'},
                        'stage2': {k: v for k, v in S2_FINAL_META.items() if k != 'history'}},
    'limitations': ['Stage 1 earthquake/flood classes each come from a single event (location confound).',
                    'Stage 2 ViT Grad-CAM on 64 px crops is a coarse 4x4 map.',
                    'Severity probabilities are not calibrated beyond training.'],
}
reqs = [f"torch>={torch.__version__.split('+')[0]}", f"torchvision>={torchvision.__version__.split('+')[0]}",
        f"timm=={timm.__version__}", f"numpy>={np.__version__}", "pillow>=9.0"]
with open(os.path.join(B, 'requirements.txt'), 'w') as fh:
    fh.write("\n".join(reqs) + "\n")
for name, obj in [('model_config.json', model_config), ('architecture_config.json', architecture_config),
                  ('label_maps.json', label_maps), ('preprocessing_config.json', bundle_pre),
                  ('inference_schema.json', inference_schema), ('model_metadata.json', model_metadata)]:
    jdump(obj, os.path.join(B, name))

README = f"""# Disaster XAI-Net — ConvNeXt + ViT trained fusion ensemble: inference bundle

Model `{MODEL_NAME}` version `{MODEL_VERSION}`. Two stages, each a jointly fine-tuned
ConvNeXt-siamese ⊕ ViT-siamese feature-fusion network:

* **Stage 1** — hazard of a pre/post tile pair: {', '.join(HAZARD_NAMES)}
* **Stage 2** — damage of each building (pre/post crop pair): {', '.join(DAMAGE_NAMES)};
  `P(damaged) = P(minor) + P(major) + P(destroyed)`, `damaged = P(damaged) > 0.5`

## Install
```
pip install -r requirements.txt
```

## Use
```python
from inference import load_models, predict_hazard, predict_damage, predict_full
load_models()                                     # or load_models(device="cpu")
predict_hazard("tile_pre.png", "tile_post.png")
predict_damage(pre_crop_img, post_crop_img)       # one building, already cropped
predict_full("tile_pre.png", "tile_post.png",
             buildings=[{{"building_id": "b1", "wkt": "POLYGON ((10 10, 40 10, 40 40, 10 40, 10 10))"}}])
```
CLI: `python inference.py pre.png post.png --labels post_disaster.json`

## Contract
* Inputs: file path, PIL image, HxWx3 uint8 array or bytes. RGB.
* Geometry: image pixels of the full-resolution tile (origin top-left). Buildings are cropped
  with a 25% padded bounding box and resized to {S2_CROP}x{S2_CROP}, exactly as in training.
* Stage 1 preprocessing: resize 256x256 → centre crop {S1_IMG} → ImageNet normalisation.
* **No latitude/longitude is computed.** `lat`/`lon` are passed through only if you supply them.
* Full output schema: `inference_schema.json`; example: `sample_prediction.json`.

## Files
| file | content |
|---|---|
| `checkpoints/stage*_ensemble_final.pth` | final weights (`state_dict`) + metadata (`meta`) |
| `model_config.json` | stages, checkpoints, version |
| `architecture_config.json` | branches, fusion head, feature dimensions |
| `label_maps.json` | class index ↔ name |
| `preprocessing_config.json` | exact preprocessing |
| `inference_schema.json` | I/O schema |
| `model_metadata.json` | CV + held-out test metrics, fold checksums, limitations |
"""
with open(os.path.join(B, 'README_INFERENCE.md'), 'w') as fh:
    fh.write(README)
print("bundle written:", sorted(os.listdir(B)))

In [ ]:
# Import the bundle exactly as the frontend backend will, and run it on one held-out test tile.
gc.collect(); torch.cuda.empty_cache()
_spec = importlib.util.spec_from_file_location("dxn_inference", os.path.join(B, "inference.py"))
dxn = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(dxn)
dxn.load_models(B)

_counts = test_sel.groupby('event_prefix').size()
_tile_id = _counts.idxmax()
_tile = test_df.set_index('event_prefix').loc[_tile_id]
_blds = test_sel[test_sel['event_prefix'] == _tile_id]
_req = [{'building_id': r.crop_id, 'uid': r.uid, 'wkt': r.polygon_wkt_xy,
         'lat': r.lat_centroid if pd.notna(r.lat_centroid) else None,
         'lon': r.lon_centroid if pd.notna(r.lon_centroid) else None} for r in _blds.itertuples()]
sample_pred = dxn.predict_full(_tile['pre_path'], _tile['post_path'], buildings=_req)
sample_pred['request'] = {'parent_image_id': _tile_id, 'pre_image': os.path.basename(_tile['pre_path']),
                          'post_image': os.path.basename(_tile['post_path']), 'n_buildings_requested': len(_req)}
jdump(sample_pred, os.path.join(B, 'sample_prediction.json'))

# consistency: bundle vs notebook test predictions for the same buildings
_nb = s2_test_pred.set_index('crop_id').loc[_blds['crop_id']]
_agree = np.mean([b['severity'] == p for b, p in zip(sample_pred['buildings'], _nb['prediction'])])
_maxdiff = np.max([abs(b['probabilities'][n] - _nb.iloc[i][f'prob_{n}'])
                   for i, b in enumerate(sample_pred['buildings']) for n in DAMAGE_NAMES])
_hz = s1_test_pred.set_index('image_id').loc[_tile_id, 'prediction']
print(f"sample tile {_tile_id}: {len(_req)} buildings | hazard bundle={sample_pred['hazard']['hazard']} "
      f"notebook={_hz} | severity agreement {_agree:.3f} | max |Δp| {_maxdiff:.4f}")
assert _agree >= 0.9 or _maxdiff < 0.05, "deployment bundle disagrees with the notebook's test predictions"
_dmg = dxn.predict_damage(Image.fromarray(t_pre[_blds['slot'].iloc[0]]), Image.fromarray(t_post[_blds['slot'].iloc[0]]))
print("predict_damage on a stored test crop:", _dmg['severity'], f"{_dmg['confidence']:.3f}")
SAMPLE_TILE_ID = _tile_id
dxn._STATE.clear(); del dxn; gc.collect(); torch.cuda.empty_cache()

---
## 14. Frontend artifacts

Per-building predictions of the **final ensemble on the held-out test set**, with every identifier and geometry the labels provide: building/crop id, xBD `uid`, parent image id, event, ground-truth and predicted hazard, image-space bbox, the 25%-padded crop box, the pixel polygon (WKT), tile size, severity, confidence, all class probabilities and `P(damaged)`. Latitude/longitude columns are filled **only** from the `lng_lat` polygons in the label files (vertex mean) and are `null` otherwise — never estimated.

In [ ]:
FE = D['frontend']
geo = test_sel.set_index('crop_id')
fb = s2_test_pred.copy()
fb['bbox_x0'] = geo.loc[fb['crop_id'], 'x0'].values; fb['bbox_y0'] = geo.loc[fb['crop_id'], 'y0'].values
fb['bbox_x1'] = geo.loc[fb['crop_id'], 'x1'].values; fb['bbox_y1'] = geo.loc[fb['crop_id'], 'y1'].values
for c in ['crop_x0', 'crop_y0', 'crop_x1', 'crop_y1']:
    fb[c] = geo.loc[fb['crop_id'], c].values
fb['polygon_wkt_xy'] = geo.loc[fb['crop_id'], 'polygon_wkt_xy'].values
fb['image_width']  = [TILE_SIZES[p][0] for p in fb['parent_image_id']]
fb['image_height'] = [TILE_SIZES[p][1] for p in fb['parent_image_id']]
fb['lat'] = geo.loc[fb['crop_id'], 'lat_centroid'].values
fb['lon'] = geo.loc[fb['crop_id'], 'lon_centroid'].values
fb['lnglat_polygon_wkt'] = geo.loc[fb['crop_id'], 'lnglat_wkt'].values
fb['geo_source'] = np.where(pd.notna(fb['lat']), 'xBD label lng_lat polygon (vertex mean)', None)
fb['coordinate_system'] = 'image_pixels'
hz = s1_test_pred.set_index('image_id')
fb['predicted_hazard_confidence'] = hz.loc[fb['parent_image_id'], 'confidence'].values
fb = fb.rename(columns={'crop_id': 'building_id', 'hazard': 'hazard_ground_truth',
                        'ground_truth': 'severity_ground_truth', 'prediction': 'severity_prediction',
                        'confidence': 'severity_confidence'})
fb['split'] = 'held_out_test'
fb['model_version'] = MODEL_VERSION
fb.to_csv(os.path.join(FE, "frontend_building_predictions.csv"), index=False)
_recs = json.loads(fb.to_json(orient='records'))
jdump({'model': {'name': MODEL_NAME, 'version': MODEL_VERSION}, 'coordinate_system': 'image_pixels',
       'geo_note': 'lat/lon only from label lng_lat; null when absent; never estimated',
       'n_buildings': len(_recs), 'buildings': _recs}, os.path.join(FE, "frontend_building_predictions.json"))

tiles = []
for tid, g in fb.groupby('parent_image_id'):
    tiles.append({'parent_image_id': tid, 'event_name': g['event_name'].iloc[0],
                  'hazard_ground_truth': g['hazard_ground_truth'].iloc[0],
                  'predicted_hazard': g['predicted_hazard'].iloc[0],
                  'predicted_hazard_confidence': float(g['predicted_hazard_confidence'].iloc[0]),
                  'image_width': int(g['image_width'].iloc[0]), 'image_height': int(g['image_height'].iloc[0]),
                  'n_buildings': int(len(g)),
                  'severity_counts': {n: int((g['severity_prediction'] == n).sum()) for n in DAMAGE_NAMES},
                  'n_damaged_pred': int((g['pred_binary'] == 'damaged').sum()),
                  'mean_p_damaged': float(g['prob_damaged'].mean())})
frontend_summary = {
    'model': {'name': MODEL_NAME, 'version': MODEL_VERSION, 'type': 'ConvNeXt+ViT trained feature-fusion ensemble'},
    'generated': datetime.datetime.now().isoformat(timespec='seconds'),
    'split': 'held_out_test', 'n_tiles': int(len(test_df)), 'n_buildings': int(len(fb)),
    'class_names': {'hazard': HAZARD_NAMES, 'damage': DAMAGE_NAMES, 'binary': BINARY_NAMES},
    'predicted_hazard_counts_tiles': s1_test_pred['prediction'].value_counts().to_dict(),
    'predicted_severity_counts': fb['severity_prediction'].value_counts().to_dict(),
    'predicted_damaged_buildings': int((fb['pred_binary'] == 'damaged').sum()),
    'geo_available_fraction': float(pd.notna(fb['lat']).mean()),
    'final_test_metrics': {r['stage']: {k: r[k] for k in ['accuracy', 'f1_macro', 'roc_auc', 'mcc', 'cohen_kappa', 'qwk']
                                        if k in r} for r in final_test_metrics.to_dict('records')},
    'cv_macro_f1_mean_std': {v: cv_cmp[(cv_cmp['model'] == ENS_NAME) & (cv_cmp['view'] == v) &
                                       (cv_cmp['metric'] == 'f1_macro')]['mean ± std'].iloc[0] for v in CV_VIEWS},
    'files': {'buildings_csv': 'frontend_building_predictions.csv', 'buildings_json': 'frontend_building_predictions.json',
              'sample_api_response': 'sample_api_response.json',
              'inference': '../deployment_bundle/inference.py'},
    'tiles': tiles,
}
jdump(frontend_summary, os.path.join(FE, "frontend_summary.json"))

sample_api = {'status': 'ok', 'api_version': '1.0', 'endpoint': 'POST /predict/full',
              'request_id': hashlib.md5(f"{SAMPLE_TILE_ID}{MODEL_VERSION}".encode()).hexdigest()[:12],
              'request': sample_pred['request'], 'data': {k: v for k, v in sample_pred.items() if k != 'request'}}
jdump(sample_api, os.path.join(FE, "sample_api_response.json"))
print(f"frontend: {len(fb)} buildings on {len(tiles)} tiles | geo available for "
      f"{frontend_summary['geo_available_fraction']:.1%} | files: {sorted(os.listdir(FE))}")

---
## 15. Consolidated summary

In [ ]:
print("=" * 78)
print(f"  DISASTER XAI-NET — {ARCH_TAG.upper()} — CONSOLIDATED RESULTS")
print("=" * 78)
print(f"  branches                   : {CONV_MODEL} siamese ⊕ {VIT_MODEL} siamese -> fusion MLP")
print(f"  params (M) stage1 / stage2 : {S1_PARAMS_M} / {S2_PARAMS_M}")
print(f"  MAX_EPOCHS / patience      : {MAX_EPOCHS} / {PATIENCE}")
print(f"  development pairs / crops  : {len(dev_df)} / {len(sel)}")
print(f"  held-out test pairs / crops: {len(test_df)} / {len(test_sel)}")
print(f"  STAGE 1 FOLD CHECKSUM      : {S1_CHECKSUM} ({'= reference' if S1_MATCHES_BASELINE else '!= reference'})")
print(f"  STAGE 2 FOLD CHECKSUM      : {S2_CHECKSUM} ({'= reference' if S2_MATCHES_BASELINE else '!= reference'})")
print("-" * 78)
for f in range(N_SPLITS):
    a, b = s1_hist[f], s2_hist[f]
    print(f"    fold {f+1}: stage1 best ep {a['best_epoch']:2d} of {a['epochs_run']:2d} "
          f"({'ES' if a['early_stopped'] else 'full'})   stage2 best ep {b['best_epoch']:2d} of "
          f"{b['epochs_run']:2d} ({'ES' if b['early_stopped'] else 'full'})")
print("=" * 78)
print("\n5-FOLD CROSS-VALIDATION (ensemble)")
print("\nStage 1 hazard");               print(s1_summary.round(4).to_string(index=False))
print("\nStage 2 multi-class severity"); print(s2_summary.round(4).to_string(index=False))
print("\nStage 2 binary severity");      print(s2b_summary.round(4).to_string(index=False))
print("\nFINAL HELD-OUT TEST (ensemble)")
print(final_test_metrics.round(4).to_string(index=False))
print("\nCOMPARISON (CV mean ± std)"); print(cv_wide.to_string())

---
## 16. Manifest, reproducibility, artifact verification and ZIP

The ZIP contains **everything** under `disaster_xai_net_master_results/` — all fold/final checkpoints, the deployment bundle and the frontend files. Required artifacts are verified before zipping (execution stops if any is missing), free disk space is checked, and after writing the archive is re-opened and every required entry and every `.pth` is confirmed inside (plus a CRC check).

In [ ]:
def sha256_file(p, chunk=1 << 24):
    h = hashlib.sha256()
    with open(p, 'rb') as fh:
        for blk in iter(lambda: fh.read(chunk), b''):
            h.update(blk)
    return h.hexdigest()

# ---- reproducibility ----
try:
    import subprocess
    _freeze = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True, timeout=120).stdout
except Exception as e:
    _freeze = f"unavailable: {e}"
with open(os.path.join(D['repro'], 'pip_freeze.txt'), 'w') as fh:
    fh.write(_freeze)
env_info = {'python': sys.version, 'platform': platform.platform(), 'torch': torch.__version__,
            'torchvision': torchvision.__version__, 'timm': timm.__version__, 'numpy': np.__version__,
            'pandas': pd.__version__, 'sklearn': sklearn.__version__,
            'cuda': torch.version.cuda, 'cudnn': torch.backends.cudnn.version() if torch.cuda.is_available() else None,
            'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, 'amp': AMP,
            'seed': SEED, 'fold_checksums': FOLD_CHECKSUMS, 'dataset_root': hazard_base_path,
            'restored_from': RESTORED_FROM, 'hyperparameters': HYPERPARAMS,
            'determinism_note': 'seeded (python/numpy/torch, per-fold SEED+fold); cuDNN non-deterministic kernels '
                                'and AMP mean reruns match closely but not bit-for-bit'}
jdump(env_info, os.path.join(D['repro'], 'environment.json'))

# ---- required artifacts ----
required = []
for st, sdir in [('stage1', 'checkpoints/stage1'), ('stage2', 'checkpoints/stage2')]:
    for k in range(1, N_SPLITS + 1):
        required += [f"{sdir}/fold_{k}_best.pth", f"{sdir}/fold_{k}_last.pth",
                     f"{sdir}/fold_{k}_done.json", f"{sdir}/fold_{k}_best_val_outputs.npz"]
required += ["checkpoints/final/stage1_ensemble_final.pth", "checkpoints/final/stage2_ensemble_final.pth"]
per_fold = ["metrics.csv", "metrics.json", "classwise_metrics.csv", "classification_report.txt",
            "confusion_matrix.png", "confusion_matrix.csv", "confusion_matrix_normalized.csv",
            "roc_curves.png", "roc_curve_points.csv", "roc_auc_values.csv", "predictions.csv"]
for k in range(1, N_SPLITS + 1):
    for sub in ['stage1_hazard', 'stage2_severity_multiclass']:
        required += [f"{sub}/fold_{k}/{x}" for x in per_fold + ["training_curves.png", "training_history.csv"]]
    required += [f"stage2_severity_binary/fold_{k}/{x}" for x in per_fold]
required += ["fold_assignments/stage1_fold_assignments.csv", "fold_assignments/stage2_fold_assignments.csv",
             "fold_assignments/fold_checksums.json",
             "oof_predictions/stage1_oof_predictions.csv", "oof_predictions/stage2_oof_predictions.csv",
             "metrics/stage1_summary.csv", "metrics/stage2_multiclass_summary.csv", "metrics/stage2_binary_summary.csv",
             "metrics/stage1_fold_metrics.csv", "metrics/stage2_multiclass_fold_metrics.csv",
             "metrics/stage2_binary_fold_metrics.csv", "metrics/final_test_metrics.csv", "metrics/final_test_metrics.json",
             "final_test/stage1_test_predictions.csv", "final_test/stage2_test_predictions.csv",
             "final_test/test_probabilities.npz"]
for v in ['stage1_hazard', 'stage2_multiclass_natural', 'stage2_multiclass_capped', 'stage2_binary']:
    required += [f"final_test/{v}/{x}" for x in ["metrics.csv", "confusion_matrix.png", "confusion_matrix_normalized.csv",
                                                  "roc_curves.png", "classwise_metrics.csv"]]
for st, tag in [('stage1', 'stage1_hazard'), ('stage2', 'stage2_severity')]:
    for br in ['convnext', 'vit']:
        required += [f"xai/{st}/{br}/{tag}_{br}_gradcam.png", f"xai/{st}/{br}/{tag}_{br}_gradcam_maps.npz",
                     f"xai/{st}/{br}/{tag}_{br}_gradcam.json"]
required += ["comparison/cv_comparison_mean_std.csv", "comparison/cv_foldwise_ensemble.csv",
             "comparison/cv_foldwise_all_models.csv", "comparison/final_test_comparison.csv", "comparison/comparison.json"]
required += [f"deployment_bundle/{x}" for x in
             ["checkpoints/stage1_ensemble_final.pth", "checkpoints/stage2_ensemble_final.pth", "model_config.json",
              "model_metadata.json", "label_maps.json", "preprocessing_config.json", "architecture_config.json",
              "inference_schema.json", "inference.py", "requirements.txt", "README_INFERENCE.md", "sample_prediction.json"]]
required += [f"frontend/{x}" for x in ["frontend_summary.json", "frontend_building_predictions.csv",
                                        "frontend_building_predictions.json", "sample_api_response.json"]]
required += ["reproducibility/environment.json", "reproducibility/hyperparameters.json",
             "reproducibility/sanity_check_report.json"]
missing = [p for p in required if not os.path.exists(os.path.join(OUT, p))]
if missing:
    print(f"!! {len(missing)} required artifact(s) MISSING:")
    for m in missing: print("   ", m)
    raise RuntimeError("required artifacts missing - re-run the cells that create them before zipping")
print(f"all {len(required)} required artifacts present")

# ---- manifest ----
shutil.rmtree(os.path.join(OUT, "_sanity"), ignore_errors=True)
files = []
for root, _, fs in os.walk(OUT):
    for fn in sorted(fs):
        if fn.endswith('.tmp') or '.tmp.' in fn: continue
        full = os.path.join(root, fn); rel = os.path.relpath(full, OUT)
        files.append({'path': rel, 'bytes': os.path.getsize(full), 'sha256': sha256_file(full)})
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
manifest = {'name': 'disaster_xai_net_master_results', 'model': MODEL_NAME, 'model_version': MODEL_VERSION,
            'created': stamp, 'fold_checksums': FOLD_CHECKSUMS,
            'fold_matched_with_baselines': {'stage1': S1_MATCHES_BASELINE, 'stage2': S2_MATCHES_BASELINE},
            'n_files': len(files) + 1, 'total_bytes': int(sum(f['bytes'] for f in files)),
            'n_checkpoints': sum(f['path'].endswith('.pth') for f in files),
            'required_artifacts_verified': len(required), 'files': files}
jdump(manifest, os.path.join(OUT, "run_manifest.json"))
print(f"manifest: {manifest['n_files']} files, {manifest['total_bytes']/1e9:.2f} GB, "
      f"{manifest['n_checkpoints']} checkpoints")

# ---- zip ----
ZIP_PATH = f"{OUT.rstrip('/')}_{stamp}.zip"
free = shutil.disk_usage(os.path.dirname(ZIP_PATH)).free
need = manifest['total_bytes'] * 1.02
print(f"zip target {ZIP_PATH} | need ~{need/1e9:.2f} GB, free {free/1e9:.2f} GB")
if free < need:
    raise RuntimeError("not enough free disk for the ZIP (checkpoints are stored uncompressed). "
                       "Free space in /kaggle/working (e.g. delete old zips) and re-run this cell.")
n_written = write_zip(OUT, ZIP_PATH, "disaster_xai_net_master_results")

with zipfile.ZipFile(ZIP_PATH) as zf:
    names = set(zf.namelist())
    arc = lambda p: f"disaster_xai_net_master_results/{p}"
    miss_zip = [p for p in required + ['run_manifest.json'] if arc(p) not in names]
    pth_disk = [f['path'] for f in files if f['path'].endswith('.pth')]
    miss_pth = [p for p in pth_disk if arc(p) not in names]
    assert not miss_zip, f"ZIP is missing required entries: {miss_zip[:10]}"
    assert not miss_pth, f"ZIP is missing checkpoints: {miss_pth[:10]}"
    bad = zf.testzip()
    assert bad is None, f"corrupt entry in ZIP: {bad}"
print(f"ZIP OK: {ZIP_PATH} ({os.path.getsize(ZIP_PATH)/1e9:.2f} GB, {n_written} files, "
      f"{len(pth_disk)} checkpoints, CRC verified)")
for sec in ['checkpoints/', 'deployment_bundle/', 'frontend/', 'xai/', 'comparison/', 'final_test/']:
    print(f"   {sec:20s} {sum(1 for n in names if n.startswith(arc(sec)))} files")

---
## Notes

* **Proposed model** = the trained ConvNeXt+ViT feature-fusion ensemble. Existing ConvNeXt / ViT are baselines (not retrained). Soft voting, if shown, is an ablation baseline.
* **Same folds as the baselines** (seed 42, same split, same crop selection) — confirmed by the checksum lines; if they ever differ the comparison table is flagged as not fold-matched.
* **Resume**: rerun the notebook top-to-bottom. Completed folds load from `fold_k_done.json` + `fold_k_best.pth`; an interrupted fold resumes from `fold_k_last.pth`; finished final models and the single test evaluation are reused. `FORCE_RETRAIN = True` wipes and retrains.
* **XAI**: two separate branch explanations of the fused decision (ConvNeXt map, ViT map); no merged heatmap.
* **Stage 1 limitation**: earthquake and flood each come from one event, so hazard accuracy partly reflects location (see `stage1_hazard/accuracy_by_event.csv`).